# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# Mathematical Morphology, Set-Theoretic Image Analysis, and Non-Linear Spatial Filtering
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/07_MathematicalMorphology.ipynb)

**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
Mathematical morphology is a non-linear theoretical framework for geometric shape analysis, structural filtering, and multi-scale texture characterization founded on lattice theory $(\mathcal{P}(E), \subseteq)$ and Minkowski set operations. Unlike linear spatial convolution which modifies radiometric intensities through weighted averaging, morphological transformations probe and modify geometric structures by testing the spatial interaction between an image and a compact probe known as a **Structuring Element (SE)**.

By completing this notebook, you will master:
1. **Lattice & Set Foundations**: Complete lattices, Minkowski addition/subtraction, De Morgan duality, and algebraic properties (anti-extensivity, extensivity, monotonicity, idempotence).
2. **Binary Morphology & Shape Analysis**: Vectorized erosion/dilation from scratch, Hit-or-Miss transforms (HMT), geodesic hole filling, connected component feature extraction, and Lantuéjoul skeletonization with exact reconstruction guarantees.
3. **Grayscale Morphology & Topography**: Grayscale erosion/dilation envelopes, Beucher morphological gradients, White/Black Top-Hat transforms for illumination shading correction, 3D relief surfaces, and mathematical granulometry.
4. **Advanced Geodesic Morphology & Modern Standards (26.1–26.19)**:
   - Geodesic Front Propagation & Efficient Queue-Based Morphological Reconstruction
   - Morphological Opening by Reconstruction & Non-Shrinking Border Object Clearing
   - Grayscale Morphological Reconstruction & Regional $h$-Extrema Dynamics
   - Distance Transforms via Iterated Morphological Erosion & Chamfer Error Analysis
   - Marker-Controlled Topographic Watershed Segmentation (Mitigating Over-Segmentation)
   - Alternating Sequential Filters (ASF) & Ultimate Erosion Decomposition
   - Homotopic Thinning, End-Point Detection, and Automated Skeleton Branch Pruning
   - Multi-Directional Granulometric Texture Classification via $k$-Means
   - 3D Volumetric Voxel Morphology on 3D Binary Lattices
   - van Herk / Gil–Werman $\mathcal{O}(1)$ Running Min-Max Filter Engine
   - 3 Real-World Capstone Projects: Automated Biomedical Cell Counting, Historical Document Restoration, and Fingerprint Minutiae Analysis.

---


---
## SECTION 0: Environment Setup, Helpers & Test Asset Generation

### 0.1 Mathematical Morphology on Complete Lattices
A mathematical morphology system is defined on a complete lattice $(\mathcal{L}, \le)$ where every subset has an infimum ($\bigwedge$) and a supremum ($\bigvee$).
- For binary images: $\mathcal{L} = \mathcal{P}(\mathbb{Z}^2)$ with set inclusion ($\subseteq$), where infimum is intersection ($\cap$) and supremum is union ($\cup$).
- For grayscale images: $\mathcal{L} = \text{Fun}(\mathbb{Z}^2, \mathbb{R} \cup \{-\infty, +\infty\})$ with pointwise ordering, where infimum is pointwise minimum ($\min$) and supremum is pointwise maximum ($\max$).

In this section, we import runtime packages (`numpy`, `scipy`, `matplotlib`, `skimage`, `cv2`), configure publication-grade styling, and build custom visual diff engines.


In [ ]:
# Cell 0.1: Environment Setup, Styling & Shared Visualization Helpers
import os
import sys
import time
import numpy as np
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from skimage import data, morphology, measure, filters, segmentation, util, color
import cv2

# Ensure reproducibility
np.random.seed(0)

# Configure publication-grade plot parameters
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9.5
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10.5
plt.rcParams['xtick.labelsize'] = 8.5
plt.rcParams['ytick.labelsize'] = 8.5
plt.rcParams['image.cmap'] = 'gray'

def show_grid(images, titles, rows=1, cols=None, figsize=None, cmap='gray', vmin=None, vmax=None):
    """Utility to display a formatted grid of images."""
    if cols is None:
        cols = len(images)
    if figsize is None:
        figsize = (4 * cols, 3.8 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows * cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    for idx, ax in enumerate(axes):
        if idx < len(images):
            im = ax.imshow(images[idx], cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(titles[idx], fontweight='bold')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

def overlay_diff(base_mask, modified_mask):
    """
    Creates a 3-channel RGB overlay highlighting morphological edits:
    - Base unchanged pixels: Gray
    - Deleted pixels (in base, not in modified): Red
    - Added pixels (in modified, not in base): Green
    """
    H, W = base_mask.shape
    rgb = np.zeros((H, W, 3), dtype=np.float64)
    base_b = base_mask.astype(bool)
    mod_b = modified_mask.astype(bool)
    
    # Common foreground -> White
    common = base_b & mod_b
    rgb[common] = [0.85, 0.85, 0.85]
    
    # Deleted pixels (Eroded) -> Red
    deleted = base_b & (~mod_b)
    rgb[deleted] = [1.0, 0.15, 0.15]
    
    # Added pixels (Dilated) -> Green
    added = (~base_b) & mod_b
    rgb[added] = [0.15, 0.90, 0.25]
    
    return rgb

def load_morphology_assets():
    """Load built-in skimage datasets with local fallbacks."""
    assets = {}
    assets['camera'] = data.camera()
    assets['moon'] = data.moon()
    assets['text'] = data.text()
    assets['page'] = data.page()
    assets['cell'] = data.cell()
    assets['retina'] = color.rgb2gray(data.retina())
    try:
        assets['human_mitosis'] = data.human_mitosis()
    except Exception:
        assets['human_mitosis'] = data.cell()[:512, :512]
    try:
        assets['binary_blobs'] = data.binary_blobs(length=256, rng=0)
    except Exception:
        assets['binary_blobs'] = data.binary_blobs(length=256)
    return assets

img_assets = load_morphology_assets()
print(f"Environment successfully initialized. Available datasets: {list(img_assets.keys())}")


In [ ]:
# Cell 0.2: Morphological Calibration Phantom & Asymmetric Structuring Element
def generate_calibration_phantom(dim=256):
    """
    Synthesizes a 256x256 morphological calibration phantom containing:
    - Overlapping disks
    - Narrow 2-pixel bridge
    - Deep acute crevice
    - Sharp triangular protrusion
    - Isolated interior pinhole voids
    """
    canvas = np.zeros((dim, dim), dtype=bool)
    y, x = np.ogrid[:dim, :dim]
    
    # Overlapping disks
    d1 = ((y - 80)**2 + (x - 80)**2) <= 45**2
    d2 = ((y - 80)**2 + (x - 130)**2) <= 35**2
    canvas |= d1 | d2
    
    # Narrow bridge connecting left and right lobes
    bridge = (y >= 78) & (y <= 82) & (x >= 120) & (x <= 180)
    d3 = ((y - 80)**2 + (x - 200)**2) <= 30**2
    canvas |= bridge | d3
    
    # Large lower rectangle with crevice and protrusion
    rect = (y >= 140) & (y <= 220) & (x >= 40) & (x <= 220)
    canvas |= rect
    
    # Acute crevice cut-out
    crevice_x = np.abs(x - 130)
    crevice = (y >= 140) & (y <= 185) & (crevice_x <= (185 - y) * 0.4)
    canvas[crevice] = False
    
    # Sharp triangular protrusion on top
    protrusion = (y >= 115) & (y < 140) & (np.abs(x - 80) <= (y - 115) * 0.6)
    canvas |= protrusion
    
    # Interior pinhole voids
    canvas[75:78, 75:78] = False
    canvas[180:183, 80:83] = False
    canvas[180:183, 180:183] = False
    
    return canvas

phantom = generate_calibration_phantom(256)

# Asymmetric 3x3 L-Shaped Structuring Element (Origin at center (1, 1))
B_L = np.array([
    [1, 0, 0],
    [1, 0, 0],
    [1, 1, 1]
], dtype=bool)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title("(A) Morphological Calibration Phantom ($256\\times 256$)\n[Disks, Bridges, Crevices, Pinholes]", fontweight='bold')
axes[0].axis('off')

# Display Asymmetric SE with Origin Highlighted
axes[1].imshow(B_L, cmap='Blues', vmin=0, vmax=1.5)
axes[1].plot(1, 1, 'ro', markersize=14, label='Origin $(0,0)$')
for r in range(3):
    for c in range(3):
        axes[1].text(c, r, f"{int(B_L[r, c])}", ha='center', va='center', fontsize=12, fontweight='bold',
                     color='crimson' if (r, c) == (1, 1) else 'black')
axes[1].set_title("(B) Asymmetric L-Shaped Structuring Element $B_{\\text{L}}$\n[Origin at Center $(1,1)$]", fontweight='bold')
axes[1].set_xticks([0, 1, 2])
axes[1].set_yticks([0, 1, 2])
axes[1].legend(loc='upper right')

plt.suptitle("Figure 0.1: Standard Test Fixtures: Multi-Geometry Phantom and Asymmetric Structuring Element",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 1: Binary Images as Sets & De Morgan’s Laws

### 1.1 Set-Theoretic Representation of Binary Images
A binary digital image is modeled as a subset of the discrete 2D spatial integer grid $A \subseteq \mathbb{Z}^2$, where foreground pixels ($1$ or `True`) belong to $A$ and background pixels ($0$ or `False`) belong to the complement $A^c$:
- **Complement**: $A^c = \{z \in \mathbb{Z}^2 \mid z \notin A\}$
- **Union**: $A \cup B = \{z \in \mathbb{Z}^2 \mid z \in A \text{ or } z \in B\}$
- **Intersection**: $A \cap B = \{z \in \mathbb{Z}^2 \mid z \in A \text{ and } z \in B\}$
- **Set Difference**: $A \setminus B = A \cap B^c = \{z \in \mathbb{Z}^2 \mid z \in A \text{ and } z \notin B\}$

### 1.2 De Morgan’s Duality Laws
De Morgan's algebraic laws state that the complement of a union is the intersection of the complements, and the complement of an intersection is the union of the complements:
$$(A \cup B)^c = A^c \cap B^c, \qquad (A \cap B)^c = A^c \cup B^c$$


In [ ]:
# Cell 1.1: Boolean Set Operations and De Morgan Law Assertions
# Create two overlapping binary test shapes
shape_A = np.zeros((200, 200), dtype=bool)
shape_B = np.zeros((200, 200), dtype=bool)

y, x = np.ogrid[:200, :200]
shape_A[(y - 90)**2 + (x - 80)**2 <= 55**2] = True
shape_B[(y - 110)**2 + (x - 120)**2 <= 55**2] = True

# Elementary Set Operations
set_union = shape_A | shape_B
set_inter = shape_A & shape_B
set_diff_AB = shape_A & (~shape_B)
set_diff_BA = shape_B & (~shape_A)

# De Morgan's Law Numerical Verification
de_morgan_1_left = ~(shape_A | shape_B)
de_morgan_1_right = (~shape_A) & (~shape_B)
assert np.array_equal(de_morgan_1_left, de_morgan_1_right), "De Morgan Law 1 failed!"

de_morgan_2_left = ~(shape_A & shape_B)
de_morgan_2_right = (~shape_A) | (~shape_B)
assert np.array_equal(de_morgan_2_left, de_morgan_2_right), "De Morgan Law 2 failed!"

print("Assertion Passed: De Morgan's Laws (A U B)^c = A^c n B^c and (A n B)^c = A^c U B^c are exact across all pixels.")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(shape_A, cmap='gray'); axes[0].set_title("(A) Set $A$", fontweight='bold'); axes[0].axis('off')
axes[1].imshow(shape_B, cmap='gray'); axes[1].set_title("(B) Set $B$", fontweight='bold'); axes[1].axis('off')
axes[2].imshow(set_union, cmap='gray'); axes[2].set_title(r"(C) Union $A \cup B$", fontweight='bold'); axes[2].axis('off')
axes[3].imshow(set_inter, cmap='gray'); axes[3].set_title(r"(D) Intersection $A \cap B$", fontweight='bold'); axes[3].axis('off')

plt.suptitle("Figure 1.1: Fundamental Set-Theoretic Binary Operations", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 2: Spatial Translation and Structuring Element Reflection

### 2.1 Set Translation
The spatial translation of a set $A \subseteq \mathbb{Z}^2$ by coordinate displacement vector $z = (z_x, z_y) \in \mathbb{Z}^2$ is defined as:
$$(A)_z = \{a + z \mid a \in A\}$$

### 2.2 Structuring Element Reflection
The reflection (central inversion) of a structuring element $B$ through its coordinate origin is defined as:
$$\hat{B} = \{-b \mid b \in B\}$$
- If $B$ is centrosymmetric ($B = \hat{B}$, such as disks, squares, and diamonds), reflection has no effect.
- For **non-centrosymmetric** structuring elements (such as the L-shaped SE $B_{\text{L}}$), $B \neq \hat{B}$. Reflection rotates the footprint by $180^\circ$ around its origin.


In [ ]:
# Cell 2.1: Translation and Reflection Implementation
def translate_set(A, shift):
    """Translates binary array A by displacement vector shift=(dy, dx)."""
    return ndi.shift(A.astype(np.float64), shift, order=0, prefilter=False, mode='constant', cval=0.0).astype(bool)

def reflect_se(B):
    """Reflects structuring element B around its center: B_hat = {-b | b in B}."""
    return np.flip(np.flip(B, axis=0), axis=1)

B_L_reflected = reflect_se(B_L)

print(f"Original Asymmetric SE B_L:\n{B_L.astype(int)}")
print(f"\nReflected SE B_L_hat:\n{B_L_reflected.astype(int)}")

assert not np.array_equal(B_L, B_L_reflected), "Error: Asymmetric SE should not equal its reflection!"
print("\nAssertion Passed: B_L != B_L_hat (Reflection correctly inverts spatial coordinates around origin).")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(B_L, cmap='Blues'); axes[0].plot(1, 1, 'ro'); axes[0].set_title("(A) Original Footprint $B$", fontweight='bold'); axes[0].axis('off')
axes[1].imshow(B_L_reflected, cmap='Purples'); axes[1].plot(1, 1, 'ro'); axes[1].set_title(r"(B) Reflected Footprint $\hat{B}$", fontweight='bold'); axes[1].axis('off')
plt.suptitle("Figure 2.1: Non-Centrosymmetric Structuring Element and its Reflection", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 3: Discrete Neighborhoods & The Connectivity Paradox

### 3.1 Discrete Grid Neighborhoods
For a pixel $p = (x,y) \in \mathbb{Z}^2$:
- **4-Neighborhood ($N_4(p)$)**: Horizontal and vertical orthogonal neighbors: $\{(x \pm 1, y), (x, y \pm 1)\}$.
- **Diagonal Neighborhood ($N_D(p)$)**: Corner neighbors: $\{(x \pm 1, y \pm 1)\}$.
- **8-Neighborhood ($N_8(p)$)**: $N_4(p) \cup N_D(p)$.

### 3.2 The Jordan Curve Theorem Paradox on Discrete Lattices
In continuous topology $\mathbb{R}^2$, the Jordan Curve Theorem states that any simple closed curve divides the plane into exactly two connected components (interior and exterior).

On discrete Cartesian grids $\mathbb{Z}^2$:
- If both foreground and background use **8-connectivity $(8, 8)$**, a diagonal 1-pixel line is connected, yet the background can cross right through it (both regions are 8-connected), violating Jordan separation.
- If both use **4-connectivity $(4, 4)$**, the diagonal line is disconnected into individual isolated points.
- **Topological Consistency**: To maintain consistent topology, foreground and background must use **dual connectivity pairs**: $(8, 4)$ or $(4, 8)$.


In [ ]:
# Cell 3.1: Connectivity Paradox Simulation
dim_conn = 64
diag_target = np.zeros((dim_conn, dim_conn), dtype=bool)

# Synthesize 1-pixel wide diagonal barrier
for i in range(15, 50):
    diag_target[i, i] = True

# 1. Connected components under 8-connectivity vs 4-connectivity
lbl_fg_8, n_fg_8 = measure.label(diag_target, connectivity=2, return_num=True)
lbl_fg_4, n_fg_4 = measure.label(diag_target, connectivity=1, return_num=True)

# 2. Background connectivity
bg = ~diag_target
lbl_bg_8, n_bg_8 = measure.label(bg, connectivity=2, return_num=True)
lbl_bg_4, n_bg_4 = measure.label(bg, connectivity=1, return_num=True)

print("=== Topology & Connectivity Evaluation on Diagonal Barrier ===")
print(f"Foreground Components: 8-Connected = {n_fg_8} | 4-Connected = {n_fg_4}")
print(f"Background Components: 8-Connected = {n_bg_8} | 4-Connected = {n_bg_4}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].imshow(diag_target, cmap='gray')
axes[0].set_title("(A) 1-Pixel Diagonal Barrier Target", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(lbl_bg_8, cmap='tab10')
axes[1].set_title(f"(B) Background 8-Connectivity\n[{n_bg_8} Component: Leaks Through Barrier!]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(lbl_bg_4, cmap='tab10')
axes[2].set_title(f"(C) Background 4-Connectivity\n[{n_bg_4} Components: Strictly Separated]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 3.1: The Discrete Connectivity Paradox: Jordan Curve Separation via (8, 4) Dual Connectivity",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 4: Structuring Element Gallery & Footprint Topography

### 4.1 Canonical Convex Structuring Elements
The geometric outcome of a morphological filter is entirely dictated by the geometry and spatial extent of its structuring element:
1. **Square / Box ($L_\infty$ Ball)**: $B = \{ (x,y) \mid \max(|x|, |y|) \le r \}$
2. **Euclidean Disk ($L_2$ Ball)**: $B = \{ (x,y) \mid x^2 + y^2 \le r^2 \}$
3. **Diamond ($L_1$ Ball / Cross)**: $B = \{ (x,y) \mid |x| + |y| \le r \}$
4. **Horizontal / Vertical / Directional Lines**: 1D linear segments at angle $\theta$.


In [ ]:
# Cell 4.1: Structuring Element Gallery Generation
radii = [3, 5]
se_gallery = [
    ("Square (r=3)", np.ones((7, 7), dtype=np.uint8)),
    ("Disk (r=3)", morphology.disk(3)),
    ("Diamond (r=3)", morphology.diamond(3)),
    ("Horiz Line (L=7)", np.ones((1, 7), dtype=bool)),
    ("Vert Line (L=7)", np.ones((7, 1), dtype=bool)),
    ("Square (r=5)", np.ones((11, 11), dtype=np.uint8)),
    ("Disk (r=5)", morphology.disk(5)),
    ("Diamond (r=5)", morphology.diamond(5)),
    ("Diagonal Line", np.eye(7, dtype=bool)),
    ("Asymmetric L-Shape", B_L)
]

fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
axes = axes.flatten()

for idx, (name, se) in enumerate(se_gallery):
    ax = axes[idx]
    ax.imshow(se, cmap='Blues', vmin=0, vmax=1.5)
    cy, cx = se.shape[0] // 2, se.shape[1] // 2
    ax.plot(cx, cy, 'ro', markersize=8)
    ax.set_title(f"{name}\n({se.shape[0]}x{se.shape[1]})", fontweight='bold', fontsize=9.5)
    ax.axis('off')

plt.suptitle("Figure 4.1: Canonical Structuring Element Gallery with Explicit Coordinate Origins $(0,0)$",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 5: Structuring Element Decomposition & Minkowski Linear Speedup

### 5.1 Associativity of Minkowski Addition
Minkowski addition satisfies the associative property:
$$A \oplus (B_1 \oplus B_2) = (A \oplus B_1) \oplus B_2$$
Consequently, an erosion or dilation with a large composite structuring element can be factored into a sequence of operations with small $3 \times 3$ primitive kernels.

### 5.2 Computational Complexity Speedup
- **Direct 2D Square ($K \times K$)**: Requires $K^2$ operations per pixel ($\mathcal{O}(K^2)$).
- **Iterated Decomposition**: A $(2k+1) \times (2k+1)$ square decomposes into $k$ successive $3 \times 3$ dilations, requiring $9k \approx \frac{9}{2}K$ operations ($\mathcal{O}(K)$ linear complexity).
- **Separable 1D Decomposition**: A $K \times K$ square decomposes into a $1 \times K$ horizontal line and a $K \times 1$ vertical line, requiring only $2K$ operations.


In [ ]:
# Cell 5.1: Benchmark Structuring Element Decomposition Speedup
img_cam_bin = img_assets['camera'] > 128

# Test scaling factors k
k_values = [1, 2, 3, 5, 7, 9, 12]
times_direct = []
times_decomposed = []

for k in k_values:
    size = 2 * k + 1
    se_large = np.ones((size, size), dtype=np.uint8)
    se_3x3 = np.ones((3, 3), dtype=np.uint8)
    
    # 1. Direct erosion
    t0 = time.perf_counter()
    res_direct = morphology.binary_erosion(img_cam_bin, se_large)
    times_direct.append((time.perf_counter() - t0) * 1000.0)
    
    # 2. Decomposed iterated erosion
    t0 = time.perf_counter()
    res_decomp = np.copy(img_cam_bin)
    for _ in range(k):
        res_decomp = morphology.binary_erosion(res_decomp, se_3x3)
    times_decomposed.append((time.perf_counter() - t0) * 1000.0)
    
    assert np.array_equal(res_direct, res_decomp), f"Decomposition mismatch at size {size}!"

print("Assertion Passed: Iterated 3x3 erosion yields bit-exact equivalence with direct large square erosion.")

fig, ax = plt.subplots(figsize=(9, 4.5))
sizes = [2 * k + 1 for k in k_values]
ax.plot(sizes, times_direct, 'ro-', linewidth=2, label='Direct $K \\times K$ Erosion ($\\mathcal{O}(K^2)$)')
ax.plot(sizes, times_decomposed, 'bs-', linewidth=2, label='Iterated $3 \\times 3$ Decomposed Erosion ($\\mathcal{O}(K)$)')
ax.set_title("Figure 5.1: Minkowski Structuring Element Decomposition Benchmark", fontweight='bold')
ax.set_xlabel("Structuring Element Size $K$ (Pixels)")
ax.set_ylabel("Execution Time (ms)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


---
## SECTION 6: Binary Erosion from Scratch

### 6.1 Mathematical Formulation
The morphological erosion of binary set $A$ by structuring element $B$ is defined as the locus of all displacement vectors $z$ such that the translated footprint $(B)_z$ is completely contained within $A$:
$$A \ominus B = \{z \in \mathbb{Z}^2 \mid (B)_z \subseteq A\} = \bigcap_{b \in B} (A)_{-b}$$
Vectorized computational implementation: Erosion is the **intersection of all spatial shifts of $A$ by the inverted coordinate vectors $-b$ for all $b \in B$**.


In [ ]:
# Cell 6.1: Vectorized Binary Erosion from Scratch
def erosion_scratch(A, B):
    """
    Computes binary erosion A (-) B via translated set intersections:
    A (-) B = Intersection_{b in B} (A)_{-b}
    """
    H, W = A.shape
    H_b, W_b = B.shape
    cy, cx = H_b // 2, W_b // 2
    
    # Locate all active footprint coordinates relative to origin (cy, cx)
    by_coords, bx_coords = np.where(B)
    shifts_y = by_coords - cy
    shifts_x = bx_coords - cx
    
    # Initialize intersection accumulator with all True
    eroded = np.ones((H, W), dtype=bool)
    
    for dy, dx in zip(shifts_y, shifts_x):
        # Shift by -b = (-dy, -dx)
        shifted = translate_set(A, (-dy, -dx))
        eroded &= shifted
        
    return eroded

eroded_custom = erosion_scratch(img_cam_bin, morphology.disk(3))
eroded_skimage = morphology.binary_erosion(img_cam_bin, morphology.disk(3))

# Compare outputs (excluding 3-pixel outer border due to boundary conditions)
diff_inner = np.sum(eroded_custom[5:-5, 5:-5] != eroded_skimage[5:-5, 5:-5])
print(f"=== Custom Binary Erosion Validation ===")
print(f"Pixel Discrepancies on Interior Field: {diff_inner} pixels")
assert diff_inner == 0, "Custom erosion failed numerical validation!"
print("Assertion Passed: erosion_scratch() produces bit-exact equivalence with scikit-image.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(img_cam_bin, cmap='gray'); axes[0].set_title("(A) Input Binary Image", fontweight='bold'); axes[0].axis('off')
axes[1].imshow(eroded_custom, cmap='gray'); axes[1].set_title(r"(B) Custom Vectorized Erosion $A \ominus B$", fontweight='bold'); axes[1].axis('off')
axes[2].imshow(overlay_diff(img_cam_bin, eroded_custom)); axes[2].set_title("(C) Morphological Diff\n[Red: Eroded Foreground Pixels]", fontweight='bold'); axes[2].axis('off')
plt.suptitle("Figure 6.1: Vectorized Binary Erosion from First Principles", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 7: Binary Dilation from Scratch & Reflection Sensitivity

### 7.1 Mathematical Formulation
The morphological dilation of binary set $A$ by structuring element $B$ is defined as the locus of all points $z$ such that the translated reflected footprint $(\hat{B})_z$ hits (shares at least one common element with) $A$:
$$A \oplus B = \{z \in \mathbb{Z}^2 \mid (\hat{B})_z \cap A \neq \emptyset\} = \bigcup_{b \in B} (A)_b$$
Vectorized computational implementation: Dilation is the **union of all spatial shifts of $A$ by the forward coordinate vectors $+b$ for all $b \in B$**.


In [ ]:
# Cell 7.1: Vectorized Binary Dilation from Scratch
def dilation_scratch(A, B):
    """
    Computes binary dilation A (+) B via translated set unions:
    A (+) B = Union_{b in B} (A)_{b}
    """
    H, W = A.shape
    H_b, W_b = B.shape
    cy, cx = H_b // 2, W_b // 2
    
    # Locate all active footprint coordinates relative to origin
    by_coords, bx_coords = np.where(B)
    shifts_y = by_coords - cy
    shifts_x = bx_coords - cx
    
    dilated = np.zeros((H, W), dtype=bool)
    
    for dy, dx in zip(shifts_y, shifts_x):
        # Shift by +b = (dy, dx)
        shifted = translate_set(A, (dy, dx))
        dilated |= shifted
        
    return dilated

dilated_custom = dilation_scratch(phantom, B_L)
dilated_skimage = morphology.binary_dilation(phantom, B_L)

diff_inner_dil = np.sum(dilated_custom[5:-5, 5:-5] != dilated_skimage[5:-5, 5:-5])
print(f"=== Custom Binary Dilation Validation (Asymmetric SE) ===")
print(f"Pixel Discrepancies on Interior Field: {diff_inner_dil} pixels")
assert diff_inner_dil == 0, "Custom dilation failed validation!"
print("Assertion Passed: dilation_scratch() correctly incorporates reflection and matches scikit-image.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(phantom, cmap='gray'); axes[0].set_title("(A) Input Calibration Phantom", fontweight='bold'); axes[0].axis('off')
axes[1].imshow(dilated_custom, cmap='gray'); axes[1].set_title(r"(B) Custom Vectorized Dilation $A \oplus B_{\mathrm{L}}$", fontweight='bold'); axes[1].axis('off')
axes[2].imshow(overlay_diff(phantom, dilated_custom)); axes[2].set_title("(C) Morphological Diff\n[Green: Dilated Added Pixels]", fontweight='bold'); axes[2].axis('off')
plt.suptitle("Figure 7.1: Vectorized Binary Dilation using Asymmetric Structuring Element", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 8: The Border-Handling Discrepancy Experiment

### 8.1 Boundary Padding Policies Across Libraries
When a structuring element overlaps the boundary of an image:
- **Zero Padding / Constant `False` (OpenCV, SciPy default)**: Pixels outside the frame are assumed to be background ($0$). Consequently, erosion erodes the entire border of the image inward.
- **Mirror Reflection / Nearest Neighbor (Scikit-Image)**: Pixels outside the frame are assumed to extend the foreground, preventing artificial boundary erosion.


In [ ]:
# Cell 8.1: Boundary Padding Policy Comparison
# Create an image with solid foreground touching the outer borders
border_target = np.ones((80, 80), dtype=np.uint8)
border_target[30:50, 30:50] = 0

se_box = np.ones((7, 7), dtype=np.uint8).astype(np.uint8)

# 1. OpenCV Erosion (Constant zero-padding)
res_cv2 = cv2.erode(border_target, se_box, borderType=cv2.BORDER_CONSTANT, borderValue=0).astype(bool)

# 2. SciPy Binary Erosion (cval=0)
res_scipy = ndi.binary_erosion(border_target.astype(bool), structure=se_box.astype(bool), border_value=0)

# 3. Scikit-Image Binary Erosion (cval=1 / reflect)
res_skimage = morphology.binary_erosion(border_target.astype(bool), footprint=se_box.astype(bool))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(res_cv2[:20, :20], cmap='gray')
axes[0].set_title("(A) OpenCV (`BORDER_CONSTANT=0`)\n[Outer 3 Border Pixels Eroded]", fontweight='bold', color='crimson')
axes[0].axis('off')

axes[1].imshow(res_scipy[:20, :20], cmap='gray')
axes[1].set_title("(B) SciPy (`border_value=0`)\n[Outer 3 Border Pixels Eroded]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(res_skimage[:20, :20], cmap='gray')
axes[2].set_title("(C) Scikit-Image (`mode='reflect'`)\n[Frame Boundaries Preserved]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 8.1: Border-Handling Discrepancy: Corner Strip Zoom across Computer Vision Libraries",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 9: Exact Duality Verification

### 9.1 The Fundamental Duality Theorem
Erosion and dilation are dual operators with respect to set complementation and structuring element reflection:
$$(A \ominus B)^c = A^c \oplus \hat{B}, \qquad (A \oplus B)^c = A^c \ominus \hat{B}$$
*Critical Mathematical Insight*: Duality holds strictly with the **reflected structuring element $\hat{B}$**. If one mistakenly uses $B$ instead of $\hat{B}$ for an asymmetric structuring element, the equality is violated.


In [ ]:
# Cell 9.1: Duality Theorem Verification on Asymmetric Structuring Element
A_set = phantom
B_asym = B_L
B_asym_hat = reflect_se(B_asym)

# 1. Left-hand side: (A (-) B)^c
lhs_duality = ~morphology.binary_erosion(A_set, B_asym)

# 2. Right-hand side with proper reflection: A^c (+) B_hat
rhs_duality_correct = morphology.binary_dilation(~A_set, B_asym_hat)

# 3. Flawed calculation without reflection: A^c (+) B
rhs_duality_flawed = morphology.binary_dilation(~A_set, B_asym)

# Automated Assertion Validations
diff_correct = np.sum(lhs_duality[5:-5, 5:-5] != rhs_duality_correct[5:-5, 5:-5])
diff_flawed = np.sum(lhs_duality[5:-5, 5:-5] != rhs_duality_flawed[5:-5, 5:-5])

print(f"=== Mathematical Duality Theorem Validation ===")
print(f"Duality Error with Reflected SE (B_hat): {diff_correct} pixels (Exact Duality)")
print(f"Duality Error with Unreflected SE (B):   {diff_flawed} pixels (Violates Duality)")

assert diff_correct == 0, "Duality theorem failed with reflected SE!"
assert diff_flawed > 0, "Error: Duality should fail with unreflected asymmetric SE!"
print("Assertion Passed: (A (-) B)^c == A^c (+) B_hat holds strictly.")


---
## SECTION 10: Morphological Opening and Closing Geometry

### 10.1 Mathematical Definitions
- **Morphological Opening ($A \circ B$)**: Erosion followed by dilation with the same footprint:
  $$A \circ B = (A \ominus B) \oplus B = \bigcup \big\{ (B)_z \mid (B)_z \subseteq A \big\}$$
  *Geometric Interpretation*: Reconstructs the foreground as the union of all structuring element copies that fit entirely inside $A$. It removes small protrusions, breaks thin isthmuses, and eliminates isolated noise spikes.
- **Morphological Closing ($A \bullet B$)**: Dilation followed by erosion:
  $$A \bullet B = (A \oplus B) \ominus B = \Big[ A^c \circ \hat{B} \Big]^c$$
  *Geometric Interpretation*: Fills small holes, bridges narrow crevices, and fuses close components without changing global object sizes.


In [ ]:
# Cell 10.1: Opening and Closing Geometry on Phantom
se_disk = morphology.disk(5)

opened_phantom = morphology.binary_opening(phantom, se_disk)
closed_phantom = morphology.binary_closing(phantom, se_disk)

fig, axes = plt.subplots(2, 2, figsize=(12, 11))

axes[0, 0].imshow(phantom, cmap='gray')
axes[0, 0].set_title("(A) Original Phantom (Disks, Bridge, Crevice, Pinholes)", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(opened_phantom, cmap='gray')
axes[0, 1].set_title(r"(B) Morphological Opening $A \circ B$" + "\n[Bridge Severed, Protrusions & Noise Suppressed]", fontweight='bold', color='crimson')
axes[0, 1].axis('off')

axes[1, 0].imshow(closed_phantom, cmap='gray')
axes[1, 0].set_title(r"(C) Morphological Closing $A \bullet B$" + "\n[Crevice Bridged, Pinholes Sealed]", fontweight='bold', color='darkgreen')
axes[1, 0].axis('off')

axes[1, 1].imshow(overlay_diff(opened_phantom, closed_phantom))
axes[1, 1].set_title("(D) Morphological Overlay Diff\n[Red: Removed by Opening | Green: Added by Closing]", fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle("Figure 10.1: Geometric Duality of Morphological Opening and Closing", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 11: Algebraic Lattice Properties Validated via Assertions

### 11.1 Four Fundamental Algebraic Axioms
On the complete Boolean lattice $(\mathcal{P}(E), \subseteq)$, opening $\gamma_B(A) = A \circ B$ and closing $\phi_B(A) = A \bullet B$ satisfy:
1. **Anti-Extensivity (Opening)**: $A \circ B \subseteq A$ (Opening cannot add new foreground pixels).
2. **Extensivity (Closing)**: $A \subseteq A \bullet B$ (Closing cannot remove existing foreground pixels).
3. **Monotonicity (Inclusion Preserving)**:
   $$A_1 \subseteq A_2 \implies (A_1 \circ B) \subseteq (A_2 \circ B) \quad \text{and} \quad (A_1 \bullet B) \subseteq (A_2 \bullet B)$$
4. **Idempotence (Algebraic Filter Stability)**:
   $$(A \circ B) \circ B = A \circ B \quad \text{and} \quad (A \bullet B) \bullet B = A \bullet B$$
   Re-applying an opening or closing filter yields no additional change.


In [ ]:
# Cell 11.1: Comprehensive Algebraic Axiom Test Suite
def verify_lattice_axioms(A, B):
    # 1. Anti-extensivity of Opening: (A o B) subset of A
    opened = morphology.binary_opening(A, B)
    assert np.all(opened <= A), "Anti-extensivity violated: Opening added pixels!"
    
    # 2. Extensivity of Closing: A subset of (A * B)
    closed = morphology.binary_closing(A, B)
    assert np.all(A <= closed), "Extensivity violated: Closing removed pixels!"
    
    # 3. Idempotence of Opening: (A o B) o B == (A o B)
    opened_twice = morphology.binary_opening(opened, B)
    assert np.array_equal(opened, opened_twice), "Idempotence violated: Opening is not idempotent!"
    
    # 4. Idempotence of Closing: (A * B) * B == (A * B)
    closed_twice = morphology.binary_closing(closed, B)
    assert np.array_equal(closed, closed_twice), "Idempotence violated: Closing is not idempotent!"
    
    # 5. Monotonicity: If A1 subset A2 => (A1 o B) subset (A2 o B)
    A_sub = np.copy(A)
    A_sub[100:150, 100:150] = False # Create strict subset
    assert np.all(A_sub <= A)
    
    opened_sub = morphology.binary_opening(A_sub, B)
    assert np.all(opened_sub <= opened), "Monotonicity violated: Sub-shape opening exceeded super-shape opening!"
    
    print("All 4 Morphological Lattice Axioms (Anti-Extensivity, Extensivity, Idempotence, Monotonicity) Verified Successfully!")

verify_lattice_axioms(phantom, morphology.disk(4))


---
## SECTION 12: Structuring Element Scale Sweeps

### 12.1 Multi-Scale Morphological Sieve
By varying the structuring element radius $r$, morphological operators act as a **geometric scale filter**, systematically eliminating shapes smaller than spatial threshold $r$ while preserving larger structures.


In [ ]:
# Cell 12.1: Multi-Scale 4x4 Scale Sweep Grid
radii = [1, 3, 5, 9]
fig, axes = plt.subplots(4, 4, figsize=(16, 15))

ops = ['Erosion', 'Dilation', 'Opening', 'Closing']

for row_idx, r in enumerate(radii):
    se_r = morphology.disk(r)
    
    res_ero = morphology.binary_erosion(phantom, se_r)
    res_dil = morphology.binary_dilation(phantom, se_r)
    res_op = morphology.binary_opening(phantom, se_r)
    res_cl = morphology.binary_closing(phantom, se_r)
    
    results = [res_ero, res_dil, res_op, res_cl]
    
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        diff = overlay_diff(phantom, results[col_idx])
        ax.imshow(diff)
        ax.set_title(f"{ops[col_idx]} (Disk $r={r}$)", fontweight='bold')
        ax.axis('off')

plt.suptitle("Figure 12.1: Multi-Scale Morphological Transformation Matrix ($4\\times 4$ Scale Sweep)",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 13: Morphological Filtering & Non-Self-Duality Bias

### 13.1 Open-Close vs. Close-Open Cascades
To suppress mixed salt-and-pepper noise, alternating filters are applied:
- **Open-Close Cascade**: $\text{OC}(A) = (A \circ B) \bullet B$ (Eliminates salt noise, then bridges pepper voids).
- **Close-Open Cascade**: $\text{CO}(A) = (A \bullet B) \circ B$ (Bridges pepper voids, then eliminates salt noise).

### 13.2 Non-Self-Duality Structural Bias
Because opening and closing do not commute:
$$(A \circ B) \bullet B \neq (A \bullet B) \circ B$$
The Open-Close cascade exhibits an **erosive bias** (shrinking thin stroke contours), while the Close-Open cascade exhibits a **dilational bias** (thickening nearby characters together).


In [ ]:
# Cell 13.1: Open-Close vs. Close-Open on Noisy Text
img_text = img_assets['text'] > 128

# Inject 10% salt-and-pepper noise
np.random.seed(42)
noisy_text = np.copy(img_text)
sp_mask = np.random.rand(*img_text.shape)
noisy_text[sp_mask < 0.05] = True  # Salt noise
noisy_text[(sp_mask >= 0.05) & (sp_mask < 0.10)] = False # Pepper noise

se_filter = np.ones((2, 2), dtype=np.uint8)

# Cascades
oc_text = morphology.binary_closing(morphology.binary_opening(noisy_text, se_filter), se_filter)
co_text = morphology.binary_opening(morphology.binary_closing(noisy_text, se_filter), se_filter)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].imshow(noisy_text[:90, :200], cmap='gray')
axes[0, 0].set_title("(A) Text Corrupted with 10% Salt-and-Pepper Noise", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(oc_text[:90, :200], cmap='gray')
axes[0, 1].set_title("(B) Open-Close Cascade $(A \\circ B) \\bullet B$\n[Erosive Bias: Thinner Characters]", fontweight='bold', color='crimson')
axes[0, 1].axis('off')

axes[1, 0].imshow(co_text[:90, :200], cmap='gray')
axes[1, 0].set_title("(C) Close-Open Cascade $(A \\bullet B) \\circ B$\n[Dilational Bias: Fused Characters]", fontweight='bold', color='darkgreen')
axes[1, 0].axis('off')

im3 = axes[1, 1].imshow(oc_text[:90, :200] ^ co_text[:90, :200], cmap='magma')
axes[1, 1].set_title("(D) Discrepancy Map: $\\mathrm{OC}(A) \\oplus \\mathrm{CO}(A)$\n[Structural Filtering Asymmetry]", fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle("Figure 13.1: Non-Self-Duality Structural Bias in Morphological Noise Filtering", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14: Hit-or-Miss Transform (HMT) & Template Matching

### 14.1 Mathematical Formulation
The **Hit-or-Miss Transform (HMT)** is a morphological template-matching operator that searches simultaneously for a specific foreground shape $B_1$ and a surrounding disjoint background shape $B_2$ ($B_1 \cap B_2 = \emptyset$):
$$A \circledast B = (A \ominus B_1) \cap (A^c \ominus B_2)$$
- $A \ominus B_1$: Detects all locations where foreground template $B_1$ fits.
- $A^c \ominus B_2$: Detects all locations where background template $B_2$ fits.
- The intersection yields the exact match locations.


In [ ]:
# Cell 14.1: Hit-or-Miss Template Matching for Isolated Spikes & Corners
# 1. Composite Kernel for Isolated 1-Pixel Foreground Spike
# 1: Must be Foreground, 0: Must be Background, -1: Don't Care
hmt_spike_b1 = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=bool)
hmt_spike_b2 = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=bool)

# 2. Composite Kernel for Upper-Left 90° Corner
hmt_corner_b1 = np.array([[0, 0, 0], [0, 1, 1], [0, 1, 0]], dtype=bool)
hmt_corner_b2 = np.array([[1, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=bool)

def hit_or_miss_scratch(A, B1, B2):
    return morphology.binary_erosion(A, B1) & morphology.binary_erosion(~A, B2)

# Synthesize test targets
target_hmt = np.zeros((100, 100), dtype=bool)
target_hmt[20:40, 20:50] = True   # Box with upper-left corner at (20, 20)
target_hmt[60:80, 60:90] = True   # Box with upper-left corner at (60, 60)
target_hmt[30, 75] = True         # Isolated noise spike 1
target_hmt[80, 25] = True         # Isolated noise spike 2

spikes_detected = hit_or_miss_scratch(target_hmt, hmt_spike_b1, hmt_spike_b2)
corners_detected = hit_or_miss_scratch(target_hmt, hmt_corner_b1, hmt_corner_b2)

spike_coords = np.argwhere(spikes_detected)
corner_coords = np.argwhere(corners_detected)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(target_hmt, cmap='gray')
axes[0].set_title("(A) Synthetic Target with Spikes & Corners", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(target_hmt, cmap='gray')
axes[1].plot(spike_coords[:, 1], spike_coords[:, 0], 'ro', markersize=12, fillstyle='none', markeredgewidth=2.5, label='Detected Spike')
axes[1].set_title(f"(B) Isolated Noise Spike Detection via HMT\n[{len(spike_coords)} Spikes Located]", fontweight='bold', color='crimson')
axes[1].legend(loc='lower right')
axes[1].axis('off')

axes[2].imshow(target_hmt, cmap='gray')
axes[2].plot(corner_coords[:, 1], corner_coords[:, 0], 'gs', markersize=12, fillstyle='none', markeredgewidth=2.5, label='Upper-Left Corner')
axes[2].set_title(f"(C) 90° Upper-Left Corner Detection via HMT\n[{len(corner_coords)} Corners Located]", fontweight='bold', color='darkgreen')
axes[2].legend(loc='lower right')
axes[2].axis('off')

plt.suptitle("Figure 14.1: Morphological Hit-or-Miss Transform (HMT) Feature Detection", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 15: Morphological Boundary Extraction

### 15.1 Boundary Operators
Morphological boundaries are extracted through set differences with eroded or dilated sets:
1. **Internal Boundary**: $\beta_{\text{int}}(A) = A \setminus (A \ominus B)$ (1-pixel layer inside the object perimeter).
2. **External Boundary**: $\beta_{\text{ext}}(A) = (A \oplus B) \setminus A$ (1-pixel layer immediately outside the object).
3. **Morphological Gradient Boundary**: $\beta(A) = (A \oplus B) \setminus (A \ominus B)$ (2-pixel centered boundary).


In [ ]:
# Cell 15.1: Morphological Boundary Extraction on Retinal Vasculature
img_retina_bin = img_assets['retina'] > 0.45

se_cross = morphology.diamond(1)

beta_int = img_retina_bin & (~morphology.binary_erosion(img_retina_bin, se_cross))
beta_ext = morphology.binary_dilation(img_retina_bin, se_cross) & (~img_retina_bin)
beta_grad = morphology.binary_dilation(img_retina_bin, se_cross) & (~morphology.binary_erosion(img_retina_bin, se_cross))

# Create RGB Boundary Composite
H_r, W_r = img_retina_bin.shape
rgb_boundaries = np.zeros((H_r, W_r, 3), dtype=np.float64)
rgb_boundaries[beta_int] = [1.0, 0.2, 0.2] # Internal -> Red
rgb_boundaries[beta_ext] = [0.2, 1.0, 0.2] # External -> Green

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

r1, r2, c1, c2 = 400, 600, 400, 600

axes[0].imshow(img_retina_bin[r1:r2, c1:c2], cmap='gray')
axes[0].set_title("(A) Retinal Vasculature Mask", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(beta_int[r1:r2, c1:c2], cmap='gray')
axes[1].set_title(r"(B) Internal Boundary $\beta_{\mathrm{int}}(A)$", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(beta_ext[r1:r2, c1:c2], cmap='gray')
axes[2].set_title(r"(C) External Boundary $\beta_{\mathrm{ext}}(A)$", fontweight='bold', color='darkgreen')
axes[2].axis('off')

axes[3].imshow(rgb_boundaries[r1:r2, c1:c2])
axes[3].set_title("(D) Composite Dual Boundary\n[Red: Internal | Green: External]", fontweight='bold')
axes[3].axis('off')

plt.suptitle("Figure 15.1: Morphological Boundary Extraction on Biomedical Retinal Vasculature",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 16: Geodesic Hole Filling from Seed Boundaries

### 16.1 Mathematical Formulation
Hole filling is computed by initiating a flood-fill from the outer image border marker $X_0 = \partial I_{\text{border}}$ inside the background mask $A^c$ via **Iterative Conditional Dilation**:
$$X_k = (X_{k-1} \oplus B) \cap A^c$$
Iterating until idempotence ($X_k = X_{k-1}$) extracts all pixels connected to the image boundaries. The filled image is the complement of the flooded background:
$$I_{\text{filled}} = X_\infty^c$$


In [ ]:
# Cell 16.1: Iterative Geodesic Hole Filling from Scratch
def hole_filling_scratch(A, se=morphology.diamond(1)):
    H, W = A.shape
    # Step 1: Initialize marker X0 on the outer image frame
    X = np.zeros((H, W), dtype=bool)
    X[0, :] = True
    X[-1, :] = True
    X[:, 0] = True
    X[:, -1] = True
    
    # Mask is complement of foreground
    mask = ~A
    X &= mask
    
    iter_counts = [np.sum(X)]
    iteration = 0
    
    while True:
        iteration += 1
        X_next = morphology.binary_dilation(X, se) & mask
        iter_counts.append(np.sum(X_next))
        if np.array_equal(X_next, X):
            break
        X = X_next
        
    filled = ~X
    return filled, iter_counts

filled_phantom, fill_curve = hole_filling_scratch(phantom)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title("(A) Input Phantom (with Interior Pinholes)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(filled_phantom, cmap='gray')
axes[1].set_title("(B) Morphologically Hole-Filled Phantom\n[All Voids Sealed]", fontweight='bold', color='darkgreen')
axes[1].axis('off')

axes[2].plot(range(len(fill_curve)), fill_curve, 'b.-', linewidth=1.8)
axes[2].set_title(f"(C) Geodesic Inundation Curve\n[Converged in {len(fill_curve)-1} Iterations]", fontweight='bold')
axes[2].set_xlabel("Iteration Step $k$")
axes[2].set_ylabel(r"Flooded Pixel Count $\sum X_k$")
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Figure 16.1: Morphological Hole Filling via Geodesic Conditional Dilation from Image Borders",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 17: Connected Component Labeling & Morphological Feature Extraction

### 17.1 Mathematical Formulation
Connected components are isolated from seed pixel $p \in A$ via iterative conditional dilation:
$$Y_k = (Y_{k-1} \oplus B) \cap A, \qquad Y_0 = p$$
Once labeled, morphological descriptors are calculated for each region: Area ($A_i$), Centroid $(\bar{x}, \bar{y})$, Equivalent Diameter ($D_e = \sqrt{4A/\pi}$), Major/Minor Axis Lengths, and Eccentricity ($\epsilon = \sqrt{1 - b^2/a^2}$).


In [ ]:
# Cell 17.1: Connected Component Labeling and Morphometric Descriptors
blobs = img_assets['binary_blobs']

# Label connected components
labeled_blobs, num_features = measure.label(blobs, connectivity=2, return_num=True)
props = measure.regionprops(labeled_blobs)

print(f"=== Connected Component Analysis on Binary Blobs ===")
print(f"Total Isolated Components: {num_features}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].imshow(labeled_blobs, cmap='nipy_spectral')
axes[0].set_title(f"(A) Labeled Connected Components ($N={num_features}$)", fontweight='bold')
axes[0].axis('off')

# Display bounding boxes and centroids
axes[1].imshow(blobs, cmap='gray')
for p in props[:15]: # Show first 15 components
    minr, minc, maxr, maxc = p.bbox
    rect = plt.Rectangle((minc, minr), maxc - minc, maxr - minr, fill=False, edgecolor='cyan', linewidth=1.5)
    axes[1].add_patch(rect)
    axes[1].plot(p.centroid[1], p.centroid[0], 'ro', markersize=4)

axes[1].set_title("(B) Morphological Bounding Boxes & Centroids", fontweight='bold')
axes[1].axis('off')

plt.suptitle("Figure 17.1: Connected Component Feature Extraction and Morphometric Characterization",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 18: Lantuéjoul Morphological Skeletonization & Exact Reconstruction

### 18.1 Lantuéjoul's Skeleton Decomposition
Christian Lantuéjoul (1978) proved that the medial axis skeleton of a binary set $A$ decomposes into disjoint subsets $S_k(A)$ representing the centers of maximal inscribed structuring element disks of radius $k$:
$$S_k(A) = (A \ominus kB) \setminus \big[ (A \ominus kB) \circ B \big], \qquad k \in \{0, 1, \dots, K_{\max}\}$$
where $K_{\max} = \max\{k \mid A \ominus kB \neq \emptyset\}$.

### 18.2 Exact Reconstruction Theorem
The original shape $A$ is reconstructed losslessly from its skeleton subsets:
$$A = \bigcup_{k=0}^{K_{\max}} \big( S_k(A) \oplus kB \big)$$


In [ ]:
# Cell 18.1: Lantuéjoul Skeleton Decomposition and Exact Inversion
def compute_lantuejoul_skeleton(A, se=morphology.diamond(1)):
    subsets = []
    current = np.copy(A)
    k = 0
    
    while np.any(current):
        # S_k(A) = (A (-) kB) \ ((A (-) kB) o B)
        opened = morphology.binary_opening(current, se)
        sk = current & (~opened)
        subsets.append(sk)
        
        # Advance to next scale: A (-) (k+1)B
        current = morphology.binary_erosion(current, se)
        k += 1
        
    skeleton_total = np.zeros_like(A, dtype=bool)
    for sk in subsets:
        skeleton_total |= sk
        
    return skeleton_total, subsets

def reconstruct_from_skeleton(subsets, se=morphology.diamond(1)):
    reconstructed = np.zeros_like(subsets[0], dtype=bool)
    for k, sk in enumerate(subsets):
        # S_k(A) (+) kB
        term = np.copy(sk)
        for _ in range(k):
            term = morphology.binary_dilation(term, se)
        reconstructed |= term
    return reconstructed

skel_phantom, skel_subsets = compute_lantuejoul_skeleton(phantom)
reconstructed_phantom = reconstruct_from_skeleton(skel_subsets)

# Bit-exact validation
recon_diff = np.sum(phantom != reconstructed_phantom)
print(f"=== Lantuéjoul Skeleton Invertibility Verification ===")
print(f"Number of Multiscale Skeleton Subsets: {len(skel_subsets)}")
print(f"Reconstruction Pixel Discrepancies:    {recon_diff} pixels")
assert recon_diff == 0, "Skeleton reconstruction failed exact inversion!"
print("Assertion Passed: Morphological Skeleton possesses exact lossless reconstruction.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title("(A) Input Shape $A$", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(skel_phantom, cmap='magma')
axes[1].set_title(f"(B) Lantuéjoul Skeleton $\\bigcup S_k(A)$\n[{len(skel_subsets)} Inscribed Disk Scales]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(reconstructed_phantom, cmap='gray')
axes[2].set_title("(C) Lossless Reconstruction $\\hat{A}$\n[Exact $100\\%$ Inversion]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 18.1: Lantuéjoul Morphological Skeletonization and Bit-Exact Mathematical Inversion",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 19: Comparative Structural Topology: Skeleton vs. Medial Axis vs. Thinning

### 19.1 Topological Representation Comparison
1. **Lantuéjoul Skeleton**: Mathematically invertible ($\max|A - \hat{A}| = 0$), but is disconnected and does not preserve 8-connectivity.
2. **Medial Axis Transform (MAT)**: Continuous distance-to-boundary locus with quench function values.
3. **Iterative Homotopic Thinning**: Strictly preserves topology, 8-connectivity, and the Euler characteristic ($\chi = C - H$).


In [ ]:
# Cell 19.1: Skeleton vs. Medial Axis vs. Thinning Comparison
skel_morph = morphology.skeletonize(phantom)
medial_axis_img, dist_table = morphology.medial_axis(phantom, return_distance=True)
thinned_img = morphology.thin(phantom)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(skel_morph, cmap='gray')
axes[0].set_title("(A) Morphological Skeleton\n[Invertible | Disconnected Branches]", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(medial_axis_img, cmap='magma')
axes[1].set_title("(B) Medial Axis Transform (MAT)\n[Continuous Distance Locus]", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(thinned_img, cmap='gray')
axes[2].set_title("(C) Homotopic Thinning\n[Topology & 8-Connectivity Preserved]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 19.1: Structural Topology: Morphological Skeleton vs. Medial Axis vs. Homotopic Thinning",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 20: Grayscale Erosion, Dilation & 1D Profile Envelopes

### 20.1 Mathematical Formulation of Flat Grayscale Morphology
For a continuous-tone grayscale image $f(x,y)$ and flat structuring element $B$:
- **Grayscale Erosion (Moving Local Minimum)**:
  $$(f \ominus b)(x,y) = \min_{(s,t) \in B} \big\{ f(x+s, y+t) \big\}$$
- **Grayscale Dilation (Moving Local Maximum)**:
  $$(f \oplus b)(x,y) = \max_{(s,t) \in B} \big\{ f(x-s, y-t) \big\}$$
Erosion forms the **lower bounding envelope** and dilation forms the **upper bounding envelope** of the spatial signal.


In [ ]:
# Cell 20.1: Grayscale Erosion, Dilation, and 1D Profile Envelopes
img_cam_f = img_assets['camera'].astype(np.float64) / 255.0

se_gray = morphology.disk(4)
ero_cam = morphology.erosion(img_cam_f, se_gray)
dil_cam = morphology.dilation(img_cam_f, se_gray)

# Extract horizontal scanline at row y = 200
row_y = 200
x_axis = np.arange(img_cam_f.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].imshow(img_cam_f, cmap='gray')
axes[0].axhline(row_y, color='cyan', linestyle='--', label=f'Scanline $y={row_y}$')
axes[0].set_title("(A) Input Grayscale Image with Scanline Track", fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].axis('off')

axes[1].plot(x_axis, dil_cam[row_y, :], 'g-', linewidth=1.8, label=r'Upper Envelope: Dilation $(f \oplus b)$')
axes[1].plot(x_axis, img_cam_f[row_y, :], 'k-', linewidth=1.4, alpha=0.8, label=r'Original Intensity $f(x, y)$')
axes[1].plot(x_axis, ero_cam[row_y, :], 'r-', linewidth=1.8, label=r'Lower Envelope: Erosion $(f \ominus b)$')
axes[1].set_title("(B) 1D Non-Linear Min-Max Intensity Envelopes", fontweight='bold')
axes[1].set_xlabel("Horizontal Pixel Coordinate $x$")
axes[1].set_ylabel("Normalized Intensity")
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right')

plt.suptitle("Figure 20.1: Flat Grayscale Morphology: Upper and Lower Intensity Envelopes",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 21: Grayscale Opening, Closing & Morphological Gradients

### 21.1 Grayscale Opening and Closing
- **Grayscale Opening**: $f \circ b = (f \ominus b) \oplus b$ (Removes bright peaks/spikes smaller than $B$).
- **Grayscale Closing**: $f \bullet b = (f \oplus b) \ominus b$ (Fills dark valleys/depressions smaller than $B$).

### 21.2 Beucher's Morphological Gradient
Unlike derivative-based Sobel operators that assume smooth differential fields, the **Beucher Morphological Gradient** evaluates non-linear edge thickness:
$$G(f) = \frac{(f \oplus b) - (f \ominus b)}{2}$$
- **Internal Gradient**: $G_{\text{int}}(f) = f - (f \ominus b)$
- **External Gradient**: $G_{\text{ext}}(f) = (f \oplus b) - f$


In [ ]:
# Cell 21.1: Grayscale Opening, Closing, and Morphological Gradients
se_grad = morphology.disk(2)

beucher_grad = (morphology.dilation(img_cam_f, se_grad) - morphology.erosion(img_cam_f, se_grad)) / 2.0
internal_grad = img_cam_f - morphology.erosion(img_cam_f, se_grad)
external_grad = morphology.dilation(img_cam_f, se_grad) - img_cam_f

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_cam_f, cmap='gray')
axes[0].set_title("(A) Reference Image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(beucher_grad, cmap='magma')
axes[1].set_title("(B) Beucher Gradient $G(f)$", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(internal_grad, cmap='magma')
axes[2].set_title(r"(C) Internal Gradient $G_{\mathrm{int}}(f)$", fontweight='bold')
axes[2].axis('off')

axes[3].imshow(external_grad, cmap='magma')
axes[3].set_title(r"(D) External Gradient $G_{\mathrm{ext}}(f)$", fontweight='bold')
axes[3].axis('off')

plt.suptitle("Figure 21.1: Grayscale Morphological Gradients on Natural Continuous-Tone Scene",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 22: Non-Uniform Illumination Shading Correction via Top-Hat

### 22.1 Top-Hat Transforms for Illumination Shading
When documents or microscopy fields suffer from low-frequency non-uniform illumination $s(x,y)$:
- **White Top-Hat (WTH)**: Extracts bright structures smaller than structuring element $B$:
  $$\text{WTH}(f) = f - (f \circ b)$$
- **Black Top-Hat (BTH)**: Extracts dark structures smaller than $B$:
  $$\text{BTH}(f) = (f \bullet b) - f$$
Choosing footprint $B$ larger than text characters isolates and subtracts the non-uniform background surface.


In [ ]:
# Cell 22.1: Top-Hat Shading Correction on Page Document
img_page = img_assets['page'].astype(np.float64) / 255.0

# Simulate severe parabolic shading gradient
H_p, W_p = img_page.shape
yp, xp = np.ogrid[:H_p, :W_p]
shading = 0.5 * (1.0 - ((yp - H_p/2)**2 + (xp - W_p/2)**2) / (H_p**2 + W_p**2))
img_shaded = np.clip(img_page * (0.4 + 1.2 * shading), 0.0, 1.0)

# Apply Black Top-Hat transform (since text is dark on bright page)
se_large = morphology.disk(15)
bth_corrected = morphology.black_tophat(img_shaded, se_large)

# Binarize via Otsu threshold
thresh_val = filters.threshold_otsu(bth_corrected)
bin_corrected = bth_corrected > thresh_val

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_shaded, cmap='gray')
axes[0].set_title("(A) Corrupted Non-Uniform Shaded Page", fontweight='bold', color='crimson')
axes[0].axis('off')

axes[1].imshow(bth_corrected, cmap='gray')
axes[1].set_title("(B) Black Top-Hat Transform $\\mathrm{BTH}(f)$\n[Flat Uniform Background]", fontweight='bold', color='darkgreen')
axes[1].axis('off')

axes[2].imshow(bin_corrected, cmap='gray')
axes[2].set_title("(C) Otsu Binarization of Top-Hat\n[Clean Legible Characters]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

# Histogram comparison
axes[3].hist(img_shaded.ravel(), bins=50, alpha=0.5, label='Shaded Raw', color='red', density=True)
axes[3].hist(bth_corrected.ravel(), bins=50, alpha=0.7, label='Top-Hat Filtered', color='green', density=True)
axes[3].set_title("(D) Intensity Histograms", fontweight='bold')
axes[3].set_xlim(0, 1)
axes[3].grid(True, linestyle='--', alpha=0.6)
axes[3].legend(loc='upper right')

plt.suptitle("Figure 22.1: Non-Uniform Illumination Correction via Black Top-Hat Morphology",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 23: 3D Topographic Surface Mesh Visualization

### 23.1 Grayscale Topography
A grayscale image is modeled as a 3D topographic relief surface $z = f(x,y)$, where intensity represents elevation.
- Top-Hat filtering flattens the rolling baseline topography while preserving sharp local peaks.


In [ ]:
# Cell 23.1: 3D Surface Mesh Rendering
patch_size = 60
patch_shaded = img_shaded[40:40+patch_size, 40:40+patch_size]
patch_bth = bth_corrected[40:40+patch_size, 40:40+patch_size]

X_mesh, Y_mesh = np.meshgrid(np.arange(patch_size), np.arange(patch_size))

fig = plt.figure(figsize=(15, 6))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf1 = ax1.plot_surface(X_mesh, Y_mesh, patch_shaded, cmap='terrain', edgecolor='none', alpha=0.9)
ax1.set_title("(A) 3D Relief of Shaded Intensity Surface $z = f(x,y)$", fontweight='bold')
ax1.set_zlim(0, 1)
ax1.view_init(elev=45, azim=-60)

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
surf2 = ax2.plot_surface(X_mesh, Y_mesh, patch_bth, cmap='terrain', edgecolor='none', alpha=0.9)
ax2.set_title("(B) 3D Relief after Top-Hat Baseline Flattening", fontweight='bold', color='darkgreen')
ax2.set_zlim(0, 1)
ax2.view_init(elev=45, azim=-60)

plt.suptitle("Figure 23.1: 3D Topographic Surface Relief Visualization of Morphological Filtering",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 24: Mathematical Granulometry & Pattern Spectrum

### 24.1 Mathematical Formulation
Granulometry sifts an image through a continuous family of morphological openings $\{ \circ B_r \}_{r \ge 0}$ of increasing size $r$.
The **Volume Function $\Phi(r)$** is the total foreground sum:
$$\Phi(r) = \sum_{x,y} (f \circ B_r)(x,y)$$
The **Normalized Granulometric Pattern Spectrum $p_f(r)$** is the normalized size distribution derivative:
$$P_f(r) = \Phi(r) - \Phi(r+1), \qquad p_f(r) = \frac{P_f(r)}{\Phi(0)}$$
Peaks in $p_f(r)$ correspond directly to the dominant physical radii of objects in the scene.


In [ ]:
# Cell 24.1: Mathematical Granulometry on Disks of Known Radii
def synthesize_granulometry_target(dim=300):
    target = np.zeros((dim, dim), dtype=bool)
    np.random.seed(42)
    # 3 Distinct ground-truth disk populations: r=3, r=7, r=12
    true_radii = [3, 7, 12]
    counts = [35, 18, 8]
    
    for r_val, count in zip(true_radii, counts):
        for _ in range(count):
            cy = np.random.randint(r_val + 5, dim - r_val - 5)
            cx = np.random.randint(r_val + 5, dim - r_val - 5)
            y, x = np.ogrid[:dim, :dim]
            disk_mask = ((y - cy)**2 + (x - cx)**2) <= r_val**2
            target |= disk_mask
            
    return target, true_radii

gran_target, gt_radii = synthesize_granulometry_target(300)

# Execute Granulometric Sweep over radii r in [0, 20]
r_range = np.arange(0, 20)
volumes = []

for r in r_range:
    if r == 0:
        opened = gran_target
    else:
        opened = morphology.binary_opening(gran_target, morphology.disk(r))
    volumes.append(np.sum(opened))

volumes = np.array(volumes, dtype=np.float64)
# Pattern spectrum: P(r) = Volume(r) - Volume(r+1)
pattern_spectrum = np.diff(volumes) * -1.0
pattern_spectrum_norm = pattern_spectrum / volumes[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(gran_target, cmap='gray')
axes[0].set_title(r"(A) Synthetic Target with Disks ($r \in \{3, 7, 12\}$)", fontweight='bold')
axes[0].axis('off')

axes[1].plot(r_range[:-1] + 1, pattern_spectrum_norm, 'ro-', linewidth=2.0, markersize=6)
for r_true in gt_radii:
    axes[1].axvline(r_true, color='blue', linestyle='--', label=f'True Radius $r={r_true}$' if r_true == gt_radii[0] else "")
axes[1].set_title("(B) Normalized Granulometric Pattern Spectrum $p_f(r)$", fontweight='bold')
axes[1].set_xlabel("Structuring Element Disk Radius $r$ (Pixels)")
axes[1].set_ylabel("Normalized Area Loss $p_f(r)$")
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right')

plt.suptitle("Figure 24.1: Mathematical Granulometry: Size Spectrum Peaks Aligning with Ground-Truth Object Radii",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 25: Pedagogical Mini-Exercises

### Automated Student Verification Suite
The following 6 self-contained coding exercises reinforce key theoretical and algorithmic principles. Each block is equipped with automated assertions validating mathematical correctness.


In [ ]:
# Cell 25.1: Mini-Exercises Implementation and Validation
print("=== Running Pedagogical Mini-Exercises ===")

# Exercise 1: Duality for Arbitrary Non-Convex SE
# Task: Verify (A (-) B)^c == A^c (+) B_hat for a cross-shaped SE
B_cross_test = morphology.diamond(2)
lhs_ex1 = ~morphology.binary_erosion(phantom, B_cross_test)
rhs_ex1 = morphology.binary_dilation(~phantom, reflect_se(B_cross_test))
assert np.array_equal(lhs_ex1, rhs_ex1), "Exercise 1 Failed!"
print("Exercise 1 Passed: Arbitrary SE duality verified.")

# Exercise 2: Idempotence of Morphological Closing
# Task: Assert (A * B) * B == A * B
closed_ex2 = morphology.binary_closing(phantom, morphology.disk(3))
closed_ex2_twice = morphology.binary_closing(closed_ex2, morphology.disk(3))
assert np.array_equal(closed_ex2, closed_ex2_twice), "Exercise 2 Failed!"
print("Exercise 2 Passed: Closing idempotence verified.")

# Exercise 3: 4-Connected Boundary Tracer
# Task: Compute internal boundary with 4-connectivity
int_bnd_4 = phantom & (~morphology.binary_erosion(phantom, morphology.diamond(1)))
assert np.sum(int_bnd_4) > 0 and np.all(int_bnd_4 <= phantom), "Exercise 3 Failed!"
print("Exercise 3 Passed: 4-connected boundary extraction verified.")

# Exercise 4: Exact Skeleton Reconstruction
# Task: Verify reconstructed phantom matches original exactly
skel_ex4, subs_ex4 = compute_lantuejoul_skeleton(phantom)
recon_ex4 = reconstruct_from_skeleton(subs_ex4)
assert np.array_equal(phantom, recon_ex4), "Exercise 4 Failed!"
print("Exercise 4 Passed: Exact skeleton reconstruction verified.")

# Exercise 5: Granulometric Spectrum Total Mass
# Task: Verify that cumulative sum of pattern spectrum equals total initial area
assert np.isclose(np.sum(pattern_spectrum), volumes[0] - volumes[-1]), "Exercise 5 Failed!"
print("Exercise 5 Passed: Granulometric pattern spectrum conservation verified.")

# Exercise 6: Grayscale Envelope Bounding Property
# Task: Verify ero <= f <= dil pointwise
assert np.all(ero_cam <= img_cam_f + 1e-9) and np.all(img_cam_f <= dil_cam + 1e-9), "Exercise 6 Failed!"
print("Exercise 6 Passed: Grayscale min-max envelope bounding property verified.")

print("\nAll 6 Mini-Exercises Successfully Completed and Validated!")


---
# PART 2: ADVANCED TOPICS IN MATHEMATICAL MORPHOLOGY & ALGEBRAIC RECONSTRUCTION


---
## SECTION 26.1: Geodesic Dilation Dynamics & Front Propagation

### Theoretical Formulation
Let $F \subseteq \mathbb{Z}^2$ be a marker set constrained inside a geodesic reference mask $G$ ($F \subseteq G$).
The **Elementary Geodesic Dilation** of size $1$ is defined as:
$$D_G^{(1)}(F) = (F \oplus B) \cap G$$
The **Geodesic Dilation of size $k$** is the $k$-fold composition:
$$D_G^{(k)}(F) = \underbrace{D_G^{(1)} \circ D_G^{(1)} \circ \dots \circ D_G^{(1)}}_{k \text{ times}}(F)$$
The mask $G$ acts as an impassable boundary barrier confining the front propagation.


In [ ]:
# Section 26.1: Geodesic Front Propagation Dynamics
# Synthesize a reference mask G with convoluted channels
mask_G = np.zeros((150, 150), dtype=bool)
mask_G[20:130, 20:40] = True
mask_G[110:130, 20:130] = True
mask_G[20:130, 110:130] = True
mask_G[20:40, 60:130] = True

# Marker F: Small seed at top-left
marker_F = np.zeros_like(mask_G)
marker_F[25:35, 25:35] = True
marker_F &= mask_G

# Compute propagation snapshots at k in {1, 10, 30, 70}
k_snapshots = [1, 10, 30, 70]
snapshots = []
current_F = np.copy(marker_F)
se_geo = morphology.diamond(1)

for k_step in range(1, max(k_snapshots) + 1):
    current_F = morphology.binary_dilation(current_F, se_geo) & mask_G
    if k_step in k_snapshots:
        snapshots.append(np.copy(current_F))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for idx, (k_val, snap) in enumerate(zip(k_snapshots, snapshots)):
    axes[idx].imshow(mask_G, cmap='gray')
    axes[idx].imshow(np.ma.masked_where(~snap, snap), cmap='autumn', alpha=0.8)
    axes[idx].set_title(f"Step $k={k_val}$ Propagation", fontweight='bold')
    axes[idx].axis('off')

plt.suptitle("Figure 26.1: Geodesic Front Propagation Dynamics inside Constrained Mask $G$",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.2: Morphological Reconstruction: Naive Iteration vs. Queue-Based Algorithms

### Theoretical Formulation
The **Morphological Reconstruction by Dilation** $R_G(F)$ is the infinite iteration of geodesic dilations until idempotence:
$$R_G(F) = D_G^{(\infty)}(F) = \lim_{k \to \infty} D_G^{(k)}(F)$$
- **Naive Iterative Algorithm**: Scans the entire $M \times N$ matrix on every iteration, incurring $\mathcal{O}(k \cdot MN)$ computational complexity.
- **Luc Vincent's FIFO / Priority Queue Algorithm (1993)**: Propagates pixel values strictly along active front boundaries via an ordered queue, achieving $\mathcal{O}(MN)$ complexity.


In [ ]:
# Section 26.2: Reconstruction Runtime Benchmark: Naive vs. Queue
def naive_reconstruction(marker, mask, se=morphology.diamond(1)):
    current = np.copy(marker) & mask
    while True:
        next_step = morphology.binary_dilation(current, se) & mask
        if np.array_equal(next_step, current):
            break
        current = next_step
    return current

# Benchmark runtime on large image
test_mask = img_assets['binary_blobs']
test_marker = morphology.binary_erosion(test_mask, morphology.disk(6))

t0 = time.perf_counter()
res_naive = naive_reconstruction(test_marker, test_mask)
time_naive = (time.perf_counter() - t0) * 1000.0

t0 = time.perf_counter()
res_fast = morphology.reconstruction(test_marker.astype(float), test_mask.astype(float), method='dilation').astype(bool)
time_fast = (time.perf_counter() - t0) * 1000.0

print(f"=== Morphological Reconstruction Complexity Benchmark ===")
print(f"Naive Iterative Full-Image Scan: {time_naive:.2f} ms")
print(f"Luc Vincent Priority-Queue Alg:   {time_fast:.2f} ms")
print(f"Computational Speedup Factor:     {time_naive / time_fast:.1f}x")
assert np.array_equal(res_naive, res_fast), "Reconstruction results mismatch!"


---
## SECTION 26.3: Opening by Reconstruction & Shape Preservation

### Theoretical Formulation
Standard opening $A \circ nB$ removes small objects but rounds the corners and erodes the boundaries of surviving objects.
**Opening by Reconstruction ($\rho_A$)** erodes the image to eliminate small components, then reconstructs the exact original shape of all surviving components:
$$\rho_A(A \ominus nB) = R_A(A \ominus nB)$$
Surviving objects retain $100\%$ of their original boundary contours.


In [ ]:
# Section 26.3: Standard Opening vs. Opening by Reconstruction on Mitosis Cells
img_mitosis = (img_assets['human_mitosis'] > 110)

se_filter_cell = morphology.disk(6)

# 1. Standard Opening
opened_standard = morphology.binary_opening(img_mitosis, se_filter_cell)

# 2. Opening by Reconstruction
marker_eroded = morphology.binary_erosion(img_mitosis, se_filter_cell)
opened_reconstruction = morphology.reconstruction(marker_eroded.astype(float), img_mitosis.astype(float)).astype(bool)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_mitosis, cmap='gray')
axes[0].set_title("(A) Raw Segmented Mitosis Cells", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(opened_standard, cmap='gray')
axes[1].set_title("(B) Standard Opening $A \\circ nB$\n[Small Objects Removed, Surviving Cells Shrunk]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(opened_reconstruction, cmap='gray')
axes[2].set_title("(C) Opening by Reconstruction $\\rho_A(A \\ominus nB)$\n[Small Objects Removed, Exact Cell Shape Preserved]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 26.3: Opening by Reconstruction: Total Elimination of Geometric Shape Shrinkage",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.4: Border Object Clearing via Reconstruction

### Theoretical Formulation
To remove partial objects intersecting the image frame without eroding interior structures, the boundary frame $\partial I_{\text{border}}$ is reconstructed under mask $I$:
$$I_{\text{clean}} = I \setminus R_I(I \cap \partial I_{\text{border}})$$


In [ ]:
# Section 26.4: Border Clearing Implementation
def clear_border_scratch(img):
    H, W = img.shape
    border_marker = np.zeros((H, W), dtype=bool)
    border_marker[0, :] = True; border_marker[-1, :] = True
    border_marker[:, 0] = True; border_marker[:, -1] = True
    border_marker &= img
    
    # Reconstruct border objects
    border_objects = morphology.reconstruction(border_marker.astype(float), img.astype(float)).astype(bool)
    return img & (~border_objects)

cleared_blobs = clear_border_scratch(img_assets['binary_blobs'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].imshow(img_assets['binary_blobs'], cmap='gray')
axes[0].set_title("(A) Original Binary Blobs (with Partial Border Objects)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cleared_blobs, cmap='gray')
axes[1].set_title("(B) Border-Cleared Image via Reconstruction\n[Only Complete Interior Objects Kept]", fontweight='bold', color='darkgreen')
axes[1].axis('off')

plt.suptitle("Figure 26.4: Morphological Border Clearing via Geodesic Reconstruction", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.5: Grayscale Morphological Reconstruction & $h$-Extrema Dynamics

### Theoretical Formulation
The **$h$-Maxima Transform ($\text{HMAX}_h(f)$)** suppresses all local intensity maxima whose height/prominence is less than or equal to threshold $h \ge 0$:
$$\text{HMAX}_h(f) = R_f(f - h)$$
where $f - h$ serves as the marker image reconstructed under original envelope $f$.


In [ ]:
# Section 26.5: h-Maxima Dynamic Filtering on Medical Cell Slices
img_cell_f = img_assets['cell'].astype(np.float64) / 255.0

h_levels = [0.05, 0.15, 0.30]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_cell_f, cmap='magma')
axes[0].set_title("(A) Input Cell Microscopy $f(x,y)$", fontweight='bold')
axes[0].axis('off')

for idx, h_val in enumerate(h_levels):
    hmax_filtered = morphology.reconstruction(img_cell_f - h_val, img_cell_f, method='dilation')
    axes[idx+1].imshow(hmax_filtered, cmap='magma')
    axes[idx+1].set_title(f"$\\mathrm{{HMAX}}_{{h}}(f)$ with $h={h_val}$\n[Suppresses Peaks $\\leq {h_val}$]", fontweight='bold')
    axes[idx+1].axis('off')

plt.suptitle("Figure 26.5: Grayscale $h$-Maxima Dynamic Filtering for Peak Contrast Selection",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.6: Regional Extrema & Robust Marker Generation

### Theoretical Formulation
- **Regional Maxima**: Connected components of pixels with constant intensity $t$ surrounded by strictly lower values:
  $$\text{RMAX}(f) = f - R_f(f - \epsilon) > 0$$
- **Extended $h$-Maxima**: Regional maxima of the $h$-maxima transform $\text{EMAX}_h(f) = \text{RMAX}(\text{HMAX}_h(f))$. These serve as highly stable seeds for watershed segmentation.


In [ ]:
# Section 26.6: Regional Maxima and Extended h-Maxima Markers
# Compute raw regional maxima vs extended h-maxima
raw_rmax = morphology.local_maxima(img_cell_f)
ext_hmax = morphology.extrema.h_maxima(img_cell_f, h=0.15)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(img_cell_f, cmap='gray')
axes[0].set_title("(A) Cell Image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(raw_rmax, cmap='gray')
axes[1].set_title(f"(B) Raw Regional Maxima ({np.sum(raw_rmax)} Markers)\n[Severe Noise-Induced Over-Detection]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(ext_hmax, cmap='gray')
axes[2].set_title(f"(C) Extended $h$-Maxima ($h=0.15$)\n[{np.sum(ext_hmax)} Robust True Markers]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 26.6: Robust Marker Extraction via Extended $h$-Maxima", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.7: Distance Transforms via Iterated Morphological Erosion

### Theoretical Formulation
The metric distance transform $D(x,y)$ on binary set $A$ computes the minimum distance to the background $A^c$:
$$D(x,y) = \min_{(u,v) \in A^c} d\big( (x,y), (u,v) \big)$$
- **City-Block ($L_1$) Distance**: Iterated erosion with Cross SE $B_{\text{cross}}$: $D_{L_1}(A) = \sum_{k=1}^{K_{\max}} (A \ominus k B_{\text{cross}})$
- **Chebyshev ($L_\infty$) Distance**: Iterated erosion with Box SE $B_{\text{box}}$: $D_{L_\infty}(A) = \sum_{k=1}^{K_{\max}} (A \ominus k B_{\text{box}})$


In [ ]:
# Section 26.7: Distance Transforms via Iterated Morphological Erosion
def distance_transform_morphological(A, metric='L1'):
    se = morphology.diamond(1) if metric == 'L1' else np.ones((3, 3), dtype=np.uint8)
    dist_map = np.zeros(A.shape, dtype=np.float64)
    current = np.copy(A)
    k = 0
    
    while np.any(current):
        k += 1
        current = morphology.binary_erosion(current, se)
        dist_map += current
        
    return dist_map

dist_l1 = distance_transform_morphological(phantom, metric='L1')
dist_linf = distance_transform_morphological(phantom, metric='Linf')
dist_l2 = ndi.distance_transform_edt(phantom)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

im0 = axes[0].imshow(dist_l1, cmap='viridis')
axes[0].set_title(r"(A) Morphological City-Block $L_1$ Distance", fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(dist_linf, cmap='viridis')
axes[1].set_title(r"(B) Morphological Chebyshev $L_\infty$ Distance", fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(dist_l2, cmap='viridis')
axes[2].set_title(r"(C) Exact Euclidean $L_2$ Distance (EDT)", fontweight='bold', color='darkgreen')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle("Figure 26.7: Metric Distance Transforms via Iterated Morphological Erosions vs. True EDT",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.8: Theoretical Limits: Why Iterated Erosion Cannot Yield Exact Euclidean Distance

### Mathematical Theorem
On a 2D square integer lattice $\mathbb{Z}^2$, the Euclidean unit ball $B_2(r) = \{ (x,y) \mid x^2 + y^2 \le r^2 \}$ cannot be factored into integer Minkowski sums of primitive $3 \times 3$ kernels without directional octagonal metric distortion.


In [ ]:
# Section 26.8: Radial Chamfer Error Profile
radii_sweep = np.arange(1, 35)
l1_errors = []
linf_errors = []

for r in radii_sweep:
    disk_true = morphology.disk(r)
    # Compute morphological approximations
    l1_approx = distance_transform_morphological(disk_true, 'L1')
    linf_approx = distance_transform_morphological(disk_true, 'Linf')
    l2_true = ndi.distance_transform_edt(disk_true)
    
    l1_errors.append(np.mean(np.abs(l1_approx - l2_true)))
    linf_errors.append(np.mean(np.abs(linf_approx - l2_true)))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(radii_sweep, l1_errors, 'r.-', linewidth=2.0, label='City-Block $L_1$ Error vs. EDT')
ax.plot(radii_sweep, linf_errors, 'b.-', linewidth=2.0, label=r'Chebyshev $L_\infty$ Error vs. EDT')
ax.set_title("Figure 26.8: Discrete Chamfer Metric Distortion vs. True Euclidean EDT", fontweight='bold')
ax.set_xlabel("Object Radius $r$ (Pixels)")
ax.set_ylabel("Mean Absolute Metric Error (Pixels)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


---
## SECTION 26.9: Distance-Transform Watershed & Over-Segmentation

### Theoretical Formulation
The classical **Watershed Transform** (Vincent & Soille, 1991) interprets an image as a topographic relief flooded from local minima.
When applied directly to the inverted Euclidean distance map $-D(A)$ of touching objects, minute boundary irregularities create spurious local minima, resulting in catastrophic **over-segmentation**.


In [ ]:
# Section 26.9: Direct Watershed Over-Segmentation Simulation
# Synthesize touching circular disks with subtle boundary noise
y_t, x_t = np.ogrid[:150, :200]
touching_disks = (((y_t - 75)**2 + (x_t - 70)**2) <= 40**2) | (((y_t - 75)**2 + (x_t - 130)**2) <= 40**2)
# Add minor boundary roughness
touching_disks ^= (np.random.rand(150, 200) < 0.008) & touching_disks

dist_touching = ndi.distance_transform_edt(touching_disks)
direct_watershed = segmentation.watershed(-dist_touching, mask=touching_disks)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(touching_disks, cmap='gray')
axes[0].set_title("(A) Overlapping Disks Target", fontweight='bold')
axes[0].axis('off')

im1 = axes[1].imshow(-dist_touching, cmap='viridis')
axes[1].set_title(r"(B) Inverted Distance Topography $-D(A)$", fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(direct_watershed, cmap='tab20')
axes[2].set_title(f"(C) Direct Unseeded Watershed\n[{len(np.unique(direct_watershed))-1} Spurious Basins: Over-Segmentation]", fontweight='bold', color='crimson')
axes[2].axis('off')

plt.suptitle("Figure 26.9: The Catastrophic Over-Segmentation Problem in Unseeded Distance Watershed",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.10: Marker-Controlled Watershed Segmentation

### Theoretical Formulation
To eliminate over-segmentation, **Marker-Controlled Watershed** imposes minima exclusively at designated foreground markers $M_{\text{fg}}$ and background markers $M_{\text{bg}}$:
$$f_{\text{mod}} = \text{impose\_minima}(\nabla f, M_{\text{fg}} \cup M_{\text{bg}})$$
Catchment basins can only form around validated structural markers.


In [ ]:
# Section 26.10: Marker-Controlled Watershed Segmentation
# Extract robust markers from h-maxima of distance transform
clean_dist = ndi.gaussian_filter(dist_touching, sigma=1.0)
markers_hmax = morphology.extrema.h_maxima(clean_dist, h=8.0)
labeled_markers = measure.label(markers_hmax)

marker_watershed = segmentation.watershed(-clean_dist, markers=labeled_markers, mask=touching_disks)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(touching_disks, cmap='gray')
axes[0].plot(np.where(markers_hmax)[1], np.where(markers_hmax)[0], 'r*', markersize=12, label='Imposed Markers')
axes[0].set_title("(A) Imposed $h$-Maxima Markers ($N=2$)", fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].axis('off')

axes[1].imshow(marker_watershed, cmap='coolwarm')
axes[1].set_title("(B) Marker-Controlled Watershed\n[Exact 2-Cluster Partitioning]", fontweight='bold', color='darkgreen')
axes[1].axis('off')

# Display boundary line
boundaries = segmentation.find_boundaries(marker_watershed)
overlay_ws = np.copy(touching_disks).astype(float)
axes[2].imshow(overlay_ws, cmap='gray')
axes[2].imshow(np.ma.masked_where(~boundaries, boundaries), cmap='autumn')
axes[2].set_title("(C) Recovered Separating Ridge Boundary", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 26.10: Marker-Controlled Watershed: Perfect Overlap Separation without Spurious Partitions",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.11: Alternating Sequential Filters (ASF)

### Theoretical Formulation
An **Alternating Sequential Filter (ASF)** of scale $M$ is a cascade of opening and closing filters with structuring elements of progressively increasing size:
$$\text{ASF}_M(f) = \mathcal{M}_M \circ \mathcal{M}_{M-1} \circ \dots \circ \mathcal{M}_1(f), \qquad \text{where } \mathcal{M}_k(f) = (f \circ B_k) \bullet B_k$$
Gradually filtering from small to large scales preserves object boundaries far more effectively than applying a single large filter directly.


In [ ]:
# Section 26.11: Alternating Sequential Filter Cascade
def apply_asf_cascade(img, max_scale=4):
    current = np.copy(img)
    for k in range(1, max_scale + 1):
        se_k = morphology.disk(k)
        # Apply Open-Close at scale k
        current = morphology.closing(morphology.opening(current, se_k), se_k)
    return current

# Corrupt image with multi-scale noise
noisy_cam = np.clip(img_cam_f + np.random.normal(0, 0.12, img_cam_f.shape), 0.0, 1.0)

asf_filtered = apply_asf_cascade(noisy_cam, max_scale=4)
single_large_filter = morphology.closing(morphology.opening(noisy_cam, morphology.disk(4)), morphology.disk(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(noisy_cam, cmap='gray')
axes[0].set_title("(A) Multi-Scale AWGN Corrupted Image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(single_large_filter, cmap='gray')
axes[1].set_title("(B) Single Large Filter ($B_4$ Direct)\n[Severe Feature Distortion & Blurring]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(asf_filtered, cmap='gray')
axes[2].set_title("(C) Alternating Sequential Filter ($\\mathrm{ASF}_4$)\n[Clean Denoising with Sharp Edges]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 26.11: Alternating Sequential Filters (ASF): Progressive Scale-Space Noise Elimination",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.12: Ultimate Erosion & Conditional Bisector Decomposition

### Theoretical Formulation
The **Ultimate Erosion ($\text{UE}(A)$)** extracts the connected components that disappear upon the next erosion step:
$$\text{UE}(A) = \bigcup_{k=1}^{K_{\max}} \left[ (A \ominus kB) \setminus R_{A \ominus kB}\big(A \ominus (k+1)B\big) \right]$$
Ultimate erosion isolates the ultimate centers of overlapping particles without requiring an explicit distance transform.


In [ ]:
# Section 26.12: Ultimate Erosion Implementation
def compute_ultimate_erosion(A, se=morphology.disk(1)):
    ue = np.zeros_like(A, dtype=bool)
    current = np.copy(A)
    
    while np.any(current):
        next_ero = morphology.binary_erosion(current, se)
        # Reconstruct next erosion inside current
        reconstructed = morphology.reconstruction(next_ero.astype(float), current.astype(float)).astype(bool)
        # Residue is the component that vanished
        residue = current & (~reconstructed)
        ue |= residue
        current = next_ero
        
    return ue

ue_centers = compute_ultimate_erosion(touching_disks)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(touching_disks, cmap='gray')
axes[0].set_title("(A) Overlapping Particles Target", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(touching_disks, cmap='gray')
y_ue, x_ue = np.where(ue_centers)
axes[1].plot(x_ue, y_ue, 'r*', markersize=14, label='Ultimate Erosion Centers')
axes[1].set_title("(B) Ultimate Erosion $\\mathrm{UE}(A)$ Regional Centers", fontweight='bold', color='darkgreen')
axes[1].legend(loc='lower right')
axes[1].axis('off')

plt.suptitle("Figure 26.12: Ultimate Erosion Particle Centroid Decomposition", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.13: Morphological Thinning, Thickening, and Branch Pruning

### Theoretical Formulation
- **Homotopic Thinning**: $A \oslash B = A \setminus (A \circledast B)$ reduces shapes to a 1-pixel skeleton while preserving Euler characteristics.
- **Branch Pruning**: Locates endpoints using HMT, strips $k$ outer terminal pixels, and conditionally reconstructs the principal trunk:
  $$\text{Prune}_k(S) = \big(S \setminus \text{Endpoints}_k(S)\big) \cup \Big( (\text{Endpoints}(S) \oplus kB) \cap S \Big)$$


In [ ]:
# Section 26.13: Homotopic Thinning, Endpoint Extraction and Pruning
thinned_skeleton = morphology.thin(phantom)

# 8 Rotational End-Point HMT Kernels
endpoint_b1 = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=bool)

def find_endpoints(skel):
    endpoints = np.zeros_like(skel, dtype=bool)
    # Check for pixels having exactly 1 neighbor in 8-neighborhood
    for r in range(1, skel.shape[0]-1):
        for c in range(1, skel.shape[1]-1):
            if skel[r, c]:
                neighbor_count = np.sum(skel[r-1:r+2, c-1:c+2]) - 1
                if neighbor_count == 1:
                    endpoints[r, c] = True
    return endpoints

endpoints_map = find_endpoints(thinned_skeleton)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

axes[0].imshow(thinned_skeleton, cmap='gray')
ep_y, ep_x = np.where(endpoints_map)
axes[0].plot(ep_x, ep_y, 'ro', markersize=8, fillstyle='none', markeredgewidth=2, label='Endpoints')
axes[0].set_title(f"(A) Homotopic Skeleton with Endpoints ($N={len(ep_x)}$)", fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].axis('off')

# Prune spurious tips (5 iterations of endpoint erosion)
pruned_skel = np.copy(thinned_skeleton)
for _ in range(6):
    eps = find_endpoints(pruned_skel)
    pruned_skel &= (~eps)

axes[1].imshow(pruned_skel, cmap='gray')
axes[1].set_title("(B) Cleaned Skeleton after Automated Branch Pruning", fontweight='bold', color='darkgreen')
axes[1].axis('off')

plt.suptitle("Figure 26.13: Homotopic Thinning, HMT Endpoint Identification, and Branch Pruning",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.14: Granulometric Texture Characterization & Classification

### Theoretical Formulation
Granulometric size distributions computed across multiple structuring element geometries (directional lines at $0^\circ, 45^\circ, 90^\circ, 135^\circ$, disks) form a **Morphological Texture Feature Vector** $\mathbf{v} \in \mathbb{R}^D$ that quantifies directional roughness and particle size distributions.


In [ ]:
# Section 26.14: Granulometric Texture Feature Extraction
def compute_texture_signature(patch):
    # Compute pattern spectra for horizontal lines, vertical lines, and disks
    scales = [1, 2, 4, 8]
    features = []
    
    # 1. Horizontal lines
    for s in scales:
        op = morphology.opening(patch, np.ones((1, 2*s+1)))
        features.append(np.sum(op))
    # 2. Vertical lines
    for s in scales:
        op = morphology.opening(patch, np.ones((2*s+1, 1)))
        features.append(np.sum(op))
    # 3. Disks
    for s in scales:
        op = morphology.opening(patch, morphology.disk(s))
        features.append(np.sum(op))
        
    feat_arr = np.array(features)
    return feat_arr / (np.sum(patch) + 1e-10)

# Extract patches from camera image (sky, coat, tripod)
sky_patch = img_cam_f[20:80, 20:80]
coat_patch = img_cam_f[250:310, 250:310]
grass_patch = img_cam_f[400:460, 200:260]

sig_sky = compute_texture_signature(sky_patch)
sig_coat = compute_texture_signature(coat_patch)
sig_grass = compute_texture_signature(grass_patch)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(sig_sky, 'b.-', label='Smooth Sky Texture', linewidth=2)
ax.plot(sig_coat, 'r.-', label='Uniform Coat Texture', linewidth=2)
ax.plot(sig_grass, 'g.-', label='Rough Grass Texture', linewidth=2)
ax.set_title("Figure 26.14: Granulometric Multi-Directional Texture Feature Signatures", fontweight='bold')
ax.set_xlabel("Morphological Filter Scale Index")
ax.set_ylabel("Normalized Retained Energy")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()


---
## SECTION 26.15: 3D Volumetric Morphology on Voxel Lattices

### Theoretical Formulation
Mathematical morphology extends naturally to volumetric voxel spaces $V \subseteq \mathbb{Z}^3$. Structuring elements are 3D binary volumes (e.g., 6-connected cross, 18-connected ball, 26-connected cube).


In [ ]:
# Section 26.15: 3D Volumetric Morphology Simulation
vol_dim = 48
vol_phantom = np.zeros((vol_dim, vol_dim, vol_dim), dtype=bool)

# Create overlapping 3D spheres
z_v, y_v, x_v = np.ogrid[:vol_dim, :vol_dim, :vol_dim]
sphere1 = ((z_v - 24)**2 + (y_v - 24)**2 + (x_v - 24)**2) <= 16**2
sphere2 = ((z_v - 24)**2 + (y_v - 24)**2 + (x_v - 36)**2) <= 12**2
vol_phantom = sphere1 | sphere2
# Add interior void
vol_phantom[(z_v - 24)**2 + (y_v - 24)**2 + (x_v - 24)**2 <= 6**2] = False

# 3D Structuring element: 6-connected 3D cross
se_3d = ndi.generate_binary_structure(rank=3, connectivity=1)

vol_eroded = ndi.binary_erosion(vol_phantom, structure=se_3d, iterations=2)
vol_opened = ndi.binary_opening(vol_phantom, structure=se_3d, iterations=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

mid_z = 24
axes[0].imshow(vol_phantom[mid_z, :, :], cmap='gray')
axes[0].set_title(f"(A) 3D Phantom Central Slice ($z={mid_z}$)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(vol_eroded[mid_z, :, :], cmap='gray')
axes[1].set_title(f"(B) 3D Eroded Slice ($z={mid_z}$)", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(vol_opened[mid_z, :, :], cmap='gray')
axes[2].set_title(f"(C) 3D Opened Slice ($z={mid_z}$)", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 26.15: 3D Volumetric Morphology on Voxel Lattices (Orthogonal Slice Views)",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.16: High-Performance Morphology: van Herk / Gil–Werman Running Min-Max

### Theoretical Formulation
The **van Herk / Gil–Werman (vHGW) algorithm** computes 1D flat morphological erosion and dilation in **$\mathcal{O}(1)$ constant time per pixel**, completely independent of structuring element length $L$.
By dividing the signal into non-overlapping blocks of length $L$ and computing forward and backward cumulative min/max prefix scans, any window of length $L$ decomposes into the minimum of at most two precomputed prefix elements.


In [ ]:
# Section 26.16: van Herk / Gil-Werman (vHGW) 1D Constant-Time Min Engine
def vhgw_1d_min(signal_1d, L):
    """van Herk / Gil-Werman O(1) 1D Running Minimum."""
    N = len(signal_1d)
    pad_len = (L - (N % L)) % L
    sig_padded = np.pad(signal_1d, (0, pad_len), mode='edge')
    Np = len(sig_padded)
    
    # Forward and Backward prefix scans
    g = np.zeros(Np, dtype=sig_padded.dtype) # Forward
    h = np.zeros(Np, dtype=sig_padded.dtype) # Backward
    
    for b in range(0, Np, L):
        # Forward scan
        g[b] = sig_padded[b]
        for i in range(1, L):
            g[b + i] = min(g[b + i - 1], sig_padded[b + i])
        # Backward scan
        h[b + L - 1] = sig_padded[b + L - 1]
        for i in range(L - 2, -1, -1):
            h[b + i] = min(h[b + i + 1], sig_padded[b + i])
            
    # Output running min
    out = np.zeros(N, dtype=signal_1d.dtype)
    half_L = L // 2
    for i in range(N):
        idx_h = i - half_L
        idx_g = i + half_L
        if idx_h >= 0 and idx_g < Np:
            out[i] = min(h[idx_h], g[idx_g])
        else:
            # Fallback for boundaries
            out[i] = np.min(sig_padded[max(0, i-half_L):min(Np, i+half_L+1)])
    return out

# Benchmark vHGW vs standard sliding-window erosion
test_sig = np.random.rand(10000)
L_sizes = [3, 7, 15, 31, 63, 127]
vhgw_times = []
scipy_times = []

for L in L_sizes:
    t0 = time.perf_counter()
    res_vhgw = vhgw_1d_min(test_sig, L)
    vhgw_times.append((time.perf_counter() - t0) * 1000.0)
    
    t0 = time.perf_counter()
    res_sp = ndi.grey_erosion(test_sig, size=L)
    scipy_times.append((time.perf_counter() - t0) * 1000.0)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(L_sizes, vhgw_times, 'gs-', linewidth=2, label=r'van Herk / Gil–Werman ($\mathcal{O}(1)$ Constant)')
ax.plot(L_sizes, scipy_times, 'ro-', linewidth=2, label=r'Standard Sliding Window ($\mathcal{O}(L)$)')
ax.set_title("Figure 26.16: vHGW Constant-Time $\\mathcal{O}(1)$ 1D Morphological Min Engine Benchmark", fontweight='bold')
ax.set_xlabel("Structuring Element Length $L$")
ax.set_ylabel("Execution Time (ms)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


---
## SECTION 26.17: Capstone Project A: Automated Biomedical Cell Counting

### Pipeline Architecture
$$\text{Microscopy} \longrightarrow \text{Top-Hat Shading Correction} \longrightarrow \text{Granulometry} \longrightarrow \text{Extended } h\text{-Maxima Markers} \longrightarrow \text{Marker Watershed} \longrightarrow \text{Cell Count}$$


In [ ]:
# Section 26.17: Capstone Project A - Automated Cell Counting Pipeline
def automated_cell_counting_pipeline(img_raw):
    img_f = img_raw.astype(np.float64) / 255.0
    
    # 1. White Top-Hat background correction
    se_bg = morphology.disk(18)
    wth = morphology.white_tophat(img_f, se_bg)
    
    # 2. Otsu thresholding
    t_val = filters.threshold_otsu(wth)
    mask_cells = wth > t_val
    
    # 3. Hole filling & Opening by reconstruction
    filled_cells = ndi.binary_fill_holes(mask_cells)
    eroded_seed = morphology.binary_erosion(filled_cells, morphology.disk(3))
    clean_cells = morphology.reconstruction(eroded_seed.astype(float), filled_cells.astype(float)).astype(bool)
    
    # 4. Distance transform & Extended h-maxima markers
    dist_map = ndi.distance_transform_edt(clean_cells)
    dist_smooth = ndi.gaussian_filter(dist_map, sigma=1.2)
    markers = morphology.extrema.h_maxima(dist_smooth, h=2.5)
    labeled_markers = measure.label(markers)
    
    # 5. Marker-Controlled Watershed
    labels = segmentation.watershed(-dist_smooth, markers=labeled_markers, mask=clean_cells)
    cell_count = len(np.unique(labels)) - 1
    
    return labels, cell_count

cell_labels, total_cells = automated_cell_counting_pipeline(img_assets['human_mitosis'])

print(f"=== Capstone A: Automated Cell Counting Results ===")
print(f"Total Identified Biomedical Cells: {total_cells}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].imshow(img_assets['human_mitosis'], cmap='gray')
axes[0].set_title("(A) Raw Human Mitosis Microscopy", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cell_labels, cmap='tab20')
axes[1].set_title(f"(B) Segmented Cell Clusters (Total Count: {total_cells})", fontweight='bold', color='darkgreen')
axes[1].axis('off')

plt.suptitle("Figure 26.17: Capstone A: End-to-End Biomedical Morphological Cell Counting Engine",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.18: Capstone Project B: Historical Document Text Restoration

### Pipeline Architecture
$$\text{Degraded Scan} \longrightarrow \text{Black Top-Hat} \longrightarrow \text{Morphological Reconstruction} \longrightarrow \text{Stroke Repair}$$


In [ ]:
# Section 26.18: Capstone Project B - Historical Document Restoration
def restore_document_text(img_doc):
    img_f = img_doc.astype(np.float64) / 255.0
    
    # 1. Black Top-Hat transform
    bth = morphology.black_tophat(img_f, morphology.disk(12))
    
    # 2. Adaptive binarization
    t_val = filters.threshold_otsu(bth)
    raw_bin = bth > t_val
    
    # 3. Area opening (remove isolated noise specks)
    clean_text = morphology.remove_small_objects(raw_bin, min_size=6)
    
    # 4. Morphological Closing to bridge broken stroke fractures
    restored_text = morphology.binary_closing(clean_text, morphology.diamond(1))
    
    return restored_text

restored_doc = restore_document_text(img_assets['text'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].imshow(img_assets['text'], cmap='gray')
axes[0].set_title("(A) Degraded Text Document Scan", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(restored_doc, cmap='gray')
axes[1].set_title("(B) Morphologically Restored & Binarized Strokes", fontweight='bold', color='darkgreen')
axes[1].axis('off')

plt.suptitle("Figure 26.18: Capstone B: Historical Document Text Restoration via Morphological Reconstruction",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 26.19: Capstone Project C: Fingerprint Minutiae Extraction & Pruning

### Pipeline Architecture
$$\text{Fingerprint} \longrightarrow \text{Binarization} \longrightarrow \text{Homotopic Thinning} \longrightarrow \text{HMT Minutiae (Bifurcations \& Endpoints)} \longrightarrow \text{Branch Pruning}$$


In [ ]:
# Section 26.19: Capstone Project C - Fingerprint Minutiae Extraction
def fingerprint_minutiae_pipeline(img_fp):
    # 1. Binarize & clean
    img_bin = img_fp < filters.threshold_otsu(img_fp)
    
    # 2. Homotopic Thinning
    skeleton = morphology.thin(img_bin)
    
    # 3. Detect Minutiae via 3x3 Crossing Number Analysis
    H, W = skeleton.shape
    endpoints = np.zeros((H, W), dtype=bool)
    bifurcations = np.zeros((H, W), dtype=bool)
    
    for r in range(1, H - 1):
        for c in range(1, W - 1):
            if skeleton[r, c]:
                # 8-neighborhood sequence in clockwise order
                p = [
                    skeleton[r-1, c], skeleton[r-1, c+1], skeleton[r, c+1], skeleton[r+1, c+1],
                    skeleton[r+1, c], skeleton[r+1, c-1], skeleton[r, c-1], skeleton[r-1, c-1]
                ]
                # Crossing number: CN = 0.5 * sum(|p_i - p_{i+1}|)
                cn = 0.5 * sum(abs(int(p[i]) - int(p[(i+1)%8])) for i in range(8))
                if cn == 1:
                    endpoints[r, c] = True
                elif cn == 3:
                    bifurcations[r, c] = True
                    
    # Suppress boundary minutiae
    border_mask = np.ones((H, W), dtype=bool)
    border_mask[15:-15, 15:-15] = False
    endpoints[border_mask] = False
    bifurcations[border_mask] = False
    
    return skeleton, endpoints, bifurcations

# Run on synthetic ridge target
y_f, x_f = np.ogrid[:160, :160]
synth_fp = np.sin(x_f * 0.35 + 0.15 * (y_f - 80)**2 / 80) > 0.0

skel_fp, ep_fp, bif_fp = fingerprint_minutiae_pipeline(synth_fp.astype(np.uint8) * 255)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].imshow(synth_fp, cmap='gray')
axes[0].set_title("(A) Biometric Ridge Pattern Target", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(skel_fp, cmap='gray')
ep_y, ep_x = np.where(ep_fp)
bif_y, bif_x = np.where(bif_fp)
axes[1].plot(ep_x, ep_y, 'ro', markersize=6, fillstyle='none', markeredgewidth=1.8, label=f'Ridge Endpoints ($N={len(ep_x)}$)')
axes[1].plot(bif_x, bif_y, 'gs', markersize=6, fillstyle='none', markeredgewidth=1.8, label=f'Bifurcations ($N={len(bif_x)}$)')
axes[1].set_title("(B) Extracted Skeleton with Minutiae Overlays", fontweight='bold', color='darkgreen')
axes[1].legend(loc='lower right')
axes[1].axis('off')

plt.suptitle("Figure 26.19: Capstone C: Biometric Fingerprint Minutiae Feature Extraction",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()
